# BTCUSD opportunity model (research only)

Single-timeframe research trainer. Set `CONFIG["resolution"]` (default `10m`)
and run every cell from the top. CPU is enough; GPU is unused. Sweep TFs
separately: `5m`, `10m`, `15m`, `30m`, `1h`, `2h`, `4h`, `6h` (native except `10m` from `5m`)
with a fixed `horizon_bars` (24 or 48), so each TF holds the same number of candles
(clock time then differs). Dual-horizon HTF sweep: `30m`/`1h`/`2h`/`4h`/`6h` at both 24 and 48.

The predictor is a **parallel bake-off of five model families plus their
soft-vote ensemble**:

1. Linear: Logistic Regression + Ridge regression
2. Random Forest
3. XGBoost
4. LightGBM
5. CatBoost
6. `ENSEMBLE` = mean of the member class probabilities

Every family is trained on the same ~45 causal features computed at the signal
candle from that one series only (multi-lookback momentum in ATR units,
volatility regime, RSI, EMA distance/slope, candle anatomy, volume z-scores,
range position, time-of-day). No higher timeframe, no flattened raw windows.
Lookbacks are bar-based, so EMA_200 spans a different wall-clock per TF.

At each **closed candle** the model answers:

> If I enter at this close and hold at most `horizon_bars` candles, is there an
> exclusive profitable side?

| Label | Meaning |
| --- | --- |
| `LONG` | Long take-profit prints on an earlier candle than the long stop; short does not win |
| `SHORT` | Short take-profit prints on an earlier candle than the short stop; long does not win |
| `NO_TRADE` | Neither side wins, both win, timeout, or same-candle TP and SL |

Regression targets on the same path (ATR units, non-negative):

- `upside` = (max future high − close) / ATR
- `downside` = (close − min future low) / ATR

## Why the bracket is wide

A classifier cannot beat a bracket whose take-profit barely clears the cost.
With TP `1.5 × ATR` on 10m BTC (≈0.32% of price) against a 0.16% round trip,
the break-even win rate was ~70% versus a 17% base rate, and every family
scored exactly `−cost`. The default bracket is now TP `4.0 × ATR`, stop
`2.0 × ATR`, `horizon_bars` 24 (4 hours on 10m; clock time scales with TF): TP ≈ 5× cost, break-even win rate
≈ 46%. The train-period gate refuses to train when the median TP distance is
below `min_tp_cost_ratio × cost`.

## Decision rule (what goes live)

The live signal is **not** `argmax`. For each side the model trades only when
`P(side)` is at least a threshold chosen on calibration rows that the model
never saw, with a minimum signal count. Abstaining is the default. A candle is
only *eligible* for a signal when its own take-profit distance
`tp_atr_mult × ATR / close` is at least `min_tp_cost_ratio × cost`; confident
predictions in quiet markets are where the fixed cost flips a 2:1 bracket into
a losing one, so they are never traded. Members and the ensemble are ranked by
**walk-forward** thresholded expectancy, not by a single validation split. The
held-out test is scored once, with t-statistic, profit factor and cost
sensitivity. `ready_to_promote` is computed by a gate, never hard-coded.

This notebook is **entry-only**. It does not predict HOLD, EXIT, trailing stops,
or any open-position action. `upside` / `downside` may later inform a
deterministic TP/SL size; they do not decide whether an existing position
should be held or exited.

`AMBIGUOUS` (same-bar TP and SL) is not a fourth trading action and is scored
as a stop-out in realised P&L (conservative).

**Do not** copy the export into `agent/model_storage/`. Live fusion stays v11.
This notebook does not import `feature_store` or `scripts.colab`, and it does
not clone the repo.


## 01 Env


In [ ]:
# CELL: ENV
# Colab ships pandas/numpy; install tabular model extras.
%pip install -q --upgrade-strategy only-if-needed pyarrow requests scikit-learn xgboost lightgbm catboost

import json
import math
import pickle
import random
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, CatBoostRegressor
from lightgbm import LGBMClassifier, LGBMRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from xgboost import XGBClassifier, XGBRegressor

print("sklearn / xgboost / lightgbm / catboost ready")


   ━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/97.1 MB 211.1 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━ 49.3/97.1 MB 182.0 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━ 86.7/97.1 MB 118.0 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 97.1/97.1 MB 152.9 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 97.1/97.1 MB 152.9 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 97.1/97.1 MB 152.9 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 97.1/97.1 MB 152.9 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 97.1/97.1 MB 152.9 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.4 MB/s eta 0:00:00


sklearn / xgboost / lightgbm / catboost ready


## 02 CONFIG


In [ ]:
# CELL: CONFIG
from pathlib import Path

RESOLUTION_MINUTES: Dict[str, int] = {
    "5m": 5,
    "10m": 10,
    "15m": 15,
    "30m": 30,
    "1h": 60,
    "2h": 120,
    "4h": 240,
    "6h": 360,
}

CONFIG: Dict[str, Any] = {
    "symbol": "BTCUSD",
    "base_url": "https://api.india.delta.exchange",
    "history_days": 900,
    "resolution": "30m",
    "horizon_bars": 24,
    "seed": 42,
    # Bracket. TP must clear the round-trip cost by a wide margin, otherwise no
    # classifier can be profitable (see the notebook header).
    "tp_atr_mult": 4.0,
    "sl_atr_mult": 2.0,
    "atr_period": 14,
    "round_trip_cost_pct": 0.0016,
    # Gate: median TP distance >= ratio x cost. Signal rule: a candle is only
    # eligible when its own TP distance >= ratio x cost (quiet-market guard).
    "min_tp_cost_ratio": 4.0,
    "minority_rate": 0.15,
    "ambiguous_win_max": 0.50,
    "train_fraction": 0.70,
    "val_fraction": 0.15,
    # Decision rule: trade only when P(side) clears a threshold chosen on
    # calibration rows. Candidates are probability quantiles of the calibration set.
    "calib_fraction": 0.15,
    "min_signals": 100,
    "signal_quantiles": [0.99, 0.98, 0.97, 0.95, 0.92, 0.90, 0.85, 0.80],
    "walk_forward_folds": 4,
    # Promotion gate thresholds (held-out test + walk-forward).
    "promote_min_tstat": 2.0,
    "promote_min_wf_positive_share": 0.67,
    "cost_sensitivity_pct": [0.0010, 0.0016, 0.0020],
    # Model capacity. Deliberately regularised: ~90k noisy rows, overlapping labels.
    "n_estimators": 400,
    "max_depth": 6,
    "learning_rate": 0.03,
    "min_leaf": 200,
    "threads_per_model": 2,
    "ensemble": ["XGB"],
}


def apply_resolution_config(cfg: Dict[str, Any]) -> None:
    """Derive bar minutes, horizon wall-clock, purge, and export_dir from resolution."""
    res = str(cfg["resolution"])
    if res not in RESOLUTION_MINUTES:
        raise ValueError(
            f"Unsupported resolution {res!r}; use one of {sorted(RESOLUTION_MINUTES)}"
        )
    bar_min = int(RESOLUTION_MINUTES[res])
    horizon_bars = int(cfg["horizon_bars"])
    if bar_min <= 0 or horizon_bars <= 0:
        raise ValueError(
            f"horizon_bars={horizon_bars} and {res} bar minutes ({bar_min}) must be positive"
        )
    cfg["bar_minutes"] = bar_min
    cfg["horizon_bars"] = horizon_bars
    cfg["horizon_minutes"] = horizon_bars * bar_min
    # Purge must cover the label look-ahead so no training label peeks into val.
    cfg["purge_bars"] = horizon_bars + 1
    cfg["export_dir"] = f"export/opportunity_{res}_h{horizon_bars}"


_override = Path("opportunity_run_config.json")
if _override.is_file():
    CONFIG.update(json.loads(_override.read_text(encoding="utf-8")))
apply_resolution_config(CONFIG)
CONFIG["ensemble"] = ["XGB"]
CONFIG["export_dir"] = "export/opportunity_30m_h24_fl_sweep"
CONFIG["run_baseline_train"] = False

LABEL_NAMES: Tuple[str, ...] = ("LONG", "SHORT", "NO_TRADE")
LABEL_LONG, LABEL_SHORT, LABEL_NO_TRADE = 0, 1, 2
ENSEMBLE_NAME = "ENSEMBLE"
print(json.dumps(CONFIG, indent=2))


{
  "symbol": "BTCUSD",
  "base_url": "https://api.india.delta.exchange",
  "history_days": 900,
  "resolution": "30m",
  "horizon_bars": 24,
  "seed": 42,
  "tp_atr_mult": 4.0,
  "sl_atr_mult": 2.0,
  "atr_period": 14,
  "round_trip_cost_pct": 0.0016,
  "min_tp_cost_ratio": 4.0,
  "minority_rate": 0.15,
  "ambiguous_win_max": 0.5,
  "train_fraction": 0.7,
  "val_fraction": 0.15,
  "calib_fraction": 0.15,
  "min_signals": 100,
  "signal_quantiles": [
    0.99,
    0.98,
    0.97,
    0.95,
    0.92,
    0.9,
    0.85,
    0.8
  ],
  "walk_forward_folds": 4,
  "promote_min_tstat": 2.0,
  "promote_min_wf_positive_share": 0.67,
  "cost_sensitivity_pct": [
    0.001,
    0.0016,
    0.002
  ],
  "n_estimators": 400,
  "max_depth": 6,
  "learning_rate": 0.03,
  "min_leaf": 200,
  "threads_per_model": 2,
  "ensemble": [
    "XGB"
  ],
  "bar_minutes": 30,
  "horizon_minutes": 720,
  "purge_bars": 25,
  "export_dir": "export/opportunity_30m_h24_fl_sweep",
  "run_baseline_train": false
}


## 03 Seed


In [ ]:
# CELL: SEED
def set_seed(seed: int) -> None:
    random.seed(int(seed))
    np.random.seed(int(seed))


set_seed(int(CONFIG["seed"]))
print("seed", CONFIG["seed"])


seed 42


## 04 Fetch OHLCV


In [ ]:
# CELL: FETCH
DELTA_CANDLE_PAGE_BARS = 2000
REQUEST_DELAY_SECONDS = 0.2
EMPTY_PAGE_RETRIES = 3
MIN_OHLCV_COMPLETENESS = 0.95
_RESOLUTION_MINUTES = {
    "1m": 1,
    "3m": 3,
    "5m": 5,
    "10m": 10,
    "15m": 15,
    "30m": 30,
    "1h": 60,
    "2h": 120,
    "4h": 240,
    "6h": 360,
    "1d": 1440,
}


def _bar_seconds(resolution: str) -> int:
    if resolution not in _RESOLUTION_MINUTES:
        raise ValueError(f"Unsupported resolution {resolution!r}")
    return int(_RESOLUTION_MINUTES[resolution]) * 60


def _parse_candle_rows(payload: Any) -> List[Dict[str, Any]]:
    if not isinstance(payload, dict):
        return []
    result = payload.get("result", [])
    if isinstance(result, dict):
        result = result.get("candles", []) or []
    if not isinstance(result, list):
        return []
    return [row for row in result if isinstance(row, dict) and "time" in row]


def _candle_epoch(row: Dict[str, Any]) -> Optional[int]:
    try:
        return int(row["time"])
    except (KeyError, TypeError, ValueError):
        return None


def _request_candle_page(
    url: str,
    *,
    symbol: str,
    resolution: str,
    start_ts: int,
    end_ts: int,
) -> List[Dict[str, Any]]:
    import requests

    last_error: Optional[Exception] = None
    for attempt in range(EMPTY_PAGE_RETRIES):
        try:
            response = requests.get(
                url,
                params={
                    "symbol": symbol,
                    "resolution": resolution,
                    "start": int(start_ts),
                    "end": int(end_ts),
                },
                timeout=30,
            )
            response.raise_for_status()
            payload = response.json()
        except Exception as exc:
            last_error = exc
            time.sleep(REQUEST_DELAY_SECONDS * (attempt + 1))
            continue
        if payload.get("success") is False:
            last_error = RuntimeError(f"Delta candles error for {symbol}: {payload}")
            time.sleep(REQUEST_DELAY_SECONDS * (attempt + 1))
            continue
        rows = _parse_candle_rows(payload)
        if rows:
            return rows
        last_error = None
        time.sleep(REQUEST_DELAY_SECONDS * (attempt + 1))
    if last_error is not None:
        raise last_error
    return []


def _rows_to_frame(all_rows: List[Dict[str, Any]]) -> pd.DataFrame:
    df = pd.DataFrame(all_rows)
    expected_cols = ["time", "open", "high", "low", "close", "volume"]
    df = df[[c for c in expected_cols if c in df.columns]]
    df["time"] = pd.to_datetime(df["time"], unit="s", utc=True)
    df = df.drop_duplicates(subset="time").sort_values("time").reset_index(drop=True)
    for col in ["open", "high", "low", "close", "volume"]:
        if col in df.columns:
            df[col] = df[col].astype(float)
    return df


def fetch_candles(
    symbol: str,
    resolution: str,
    start_ts: int,
    end_ts: int,
    base_url: str,
    *,
    page_bars: int = DELTA_CANDLE_PAGE_BARS,
) -> pd.DataFrame:
    url = f"{base_url.rstrip('/')}/v2/history/candles"
    bar_seconds = _bar_seconds(resolution)
    page_bars = max(1, int(page_bars))
    start_ts, end_ts = int(start_ts), int(end_ts)
    if end_ts <= start_ts:
        raise ValueError("end_ts must be greater than start_ts")
    all_rows: List[Dict[str, Any]] = []
    cursor_end = end_ts
    seen_oldest: Optional[int] = None
    max_pages = max(2, (end_ts - start_ts) // bar_seconds + 5)
    for _ in range(max_pages):
        if cursor_end <= start_ts:
            break
        page_start = max(start_ts, cursor_end - page_bars * bar_seconds)
        if page_start >= cursor_end:
            break
        rows = _request_candle_page(
            url,
            symbol=symbol,
            resolution=resolution,
            start_ts=page_start,
            end_ts=cursor_end,
        )
        if not rows:
            break
        epochs = [e for e in (_candle_epoch(r) for r in rows) if e is not None]
        if not epochs:
            break
        oldest = min(epochs)
        if seen_oldest is not None and oldest >= seen_oldest:
            break
        seen_oldest = oldest
        all_rows.extend(rows)
        cursor_end = oldest - 1
        time.sleep(REQUEST_DELAY_SECONDS)
    if not all_rows:
        raise RuntimeError(f"No candle data returned for symbol={symbol}")
    return _rows_to_frame(all_rows)


def validate_ohlcv_completeness(
    df: pd.DataFrame,
    resolution: str,
    *,
    min_completeness: float = MIN_OHLCV_COMPLETENESS,
    symbol: str = "",
) -> Dict[str, Any]:
    if df is None or df.empty or "time" not in df.columns:
        raise ValueError("Cannot validate completeness on an empty OHLCV frame")
    bar_seconds = _bar_seconds(resolution)
    time_col = df["time"]
    epochs = (pd.to_datetime(time_col, utc=True).astype("int64") // 10**9).tolist()
    if len(epochs) < 2:
        raise ValueError("Need at least 2 OHLCV bars to validate completeness")
    gaps = 0
    for prev, cur in zip(epochs, epochs[1:]):
        if int(cur) - int(prev) > bar_seconds * 1.5:
            gaps += 1
    completeness = 1.0 - (gaps / max(1, len(epochs) - 1))
    report = {
        "rows": len(epochs),
        "gaps": int(gaps),
        "completeness": round(float(completeness), 4),
    }
    if completeness < min_completeness:
        label = f"{symbol} {resolution}".strip()
        raise ValueError(
            f"OHLCV completeness {completeness:.1%} below {min_completeness:.0%} "
            f"for {label or 'candles'}"
        )
    return report


end_dt = datetime.now(timezone.utc)
start_dt = end_dt - timedelta(days=int(CONFIG["history_days"]))
_resolution = str(CONFIG["resolution"])
# Delta has no native 10m candles; everything else is fetched at its own resolution.
_source_resolution = "5m" if _resolution == "10m" else _resolution
print(
    f"Fetching {CONFIG['symbol']} {_source_resolution} OHLCV for {CONFIG['history_days']}d "
    f"from {CONFIG['base_url']} (target {_resolution})..."
)
raw_bars = fetch_candles(
    str(CONFIG["symbol"]),
    _source_resolution,
    int(start_dt.timestamp()),
    int(end_dt.timestamp()),
    str(CONFIG["base_url"]),
)
ohlcv_report = validate_ohlcv_completeness(
    raw_bars, _source_resolution, symbol=str(CONFIG["symbol"])
)
print(
    f"OHLCV {CONFIG['symbol']} {_source_resolution}: {ohlcv_report['rows']} bars, "
    f"completeness {ohlcv_report['completeness']:.1%}, gaps={ohlcv_report['gaps']}"
)


Fetching BTCUSD 30m OHLCV for 900d from https://api.india.delta.exchange (target 30m)...


OHLCV BTCUSD 30m: 43200 bars, completeness 100.0%, gaps=0


## 05 Confirm bars


In [ ]:
# CELL: BARS
def build_10m_ohlcv_from_5m(df5m: pd.DataFrame) -> pd.DataFrame:
    """Exchange-aligned 10m bars from two closed 5m candles. Drop a partial last bucket."""
    src = df5m.copy()
    src["time"] = pd.to_datetime(src["time"], utc=True)
    required = ("open", "high", "low", "close", "volume")
    missing = [c for c in required if c not in src.columns]
    if missing:
        raise ValueError(f"5m OHLCV missing columns: {missing}")
    if src.empty:
        return src.iloc[0:0].copy()
    grouped = src.set_index("time").sort_index()
    agg = grouped.resample("10min", label="left", closed="left").agg(
        {
            "open": "first",
            "high": "max",
            "low": "min",
            "close": "last",
            "volume": "sum",
        }
    )
    counts = grouped["close"].resample("10min", label="left", closed="left").count()
    complete = counts >= 2
    agg = agg.loc[complete].dropna(subset=["open", "high", "low", "close"])
    out = agg.reset_index()
    return out.sort_values("time").reset_index(drop=True)


if str(CONFIG["resolution"]) == "10m":
    bars = build_10m_ohlcv_from_5m(raw_bars)
else:
    bars = raw_bars
if bars is None or getattr(bars, "empty", True):
    raise ValueError(f"No OHLCV bars for {CONFIG['resolution']}")
print(
    f"{CONFIG['resolution']} bars={len(bars)} "
    f"span={bars['time'].iloc[0]} → {bars['time'].iloc[-1]} "
    f"horizon_bars={CONFIG['horizon_bars']} ({CONFIG['horizon_minutes']}m)"
)


30m bars=43200 span=2024-04-25 13:30:00+00:00 → 2026-10-12 13:00:00+00:00 horizon_bars=24 (720m)


## 06 Features


In [ ]:
# CELL: FEATURES
OHLCV_COLS: Tuple[str, ...] = ("time", "open", "high", "low", "close", "volume")


def _wilder_rsi(close: pd.Series, period: int = 14) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0.0)
    loss = -delta.clip(upper=0.0)
    avg_gain = gain.ewm(alpha=1.0 / period, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1.0 / period, min_periods=period, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0.0, np.nan)
    return 100.0 - (100.0 / (1.0 + rs))


def add_opportunity_features(df: pd.DataFrame, *, atr_period: int = 14) -> pd.DataFrame:
    """Causal signal-candle features from the selected timeframe series only.

    Everything is expressed in ATR units or as a ratio so the model is scale
    invariant across the 900-day price range. ATR(14) is also stored for the
    label path. Nothing here looks past the signal candle.
    """
    out = df.copy()
    close = out["close"].astype(np.float64)
    high = out["high"].astype(np.float64)
    low = out["low"].astype(np.float64)
    open_ = out["open"].astype(np.float64)
    volume = out["volume"].astype(np.float64)
    prev_close = close.shift(1)
    tr = pd.concat(
        [high - low, (high - prev_close).abs(), (low - prev_close).abs()], axis=1
    ).max(axis=1)
    atr = tr.rolling(int(atr_period), min_periods=int(atr_period)).mean()
    atr_safe = atr.replace(0.0, np.nan)
    out["atr"] = atr

    feats: Dict[str, pd.Series] = {}
    # Momentum at several lookbacks, in ATR units.
    feats["ret_1"] = close.pct_change()
    for k in (3, 6, 12, 24, 48, 144):
        feats[f"ret_{k}_atr"] = (close - close.shift(k)) / atr_safe
    # Volatility level and regime.
    feats["atr_pct"] = atr / close.replace(0.0, np.nan)
    atr_long = tr.rolling(int(atr_period) * 6, min_periods=int(atr_period) * 6).mean()
    feats["atr_ratio_84"] = atr / atr_long.replace(0.0, np.nan)
    log_ret = np.log(close).diff()
    rv_12 = log_ret.rolling(12, min_periods=12).std()
    rv_48 = log_ret.rolling(48, min_periods=48).std()
    rv_144 = log_ret.rolling(144, min_periods=144).std()
    feats["rv_12_48"] = rv_12 / rv_48.replace(0.0, np.nan)
    feats["rv_48_144"] = rv_48 / rv_144.replace(0.0, np.nan)
    # Oscillators and trend.
    for period in (7, 14, 28):
        feats[f"rsi_{period}"] = _wilder_rsi(close, period=period)
    for span in (20, 50, 200):
        ema = close.ewm(span=span, adjust=False, min_periods=span).mean()
        feats[f"ema_dist_{span}_atr"] = (close - ema) / atr_safe
        feats[f"ema_slope_{span}_atr"] = (ema - ema.shift(6)) / atr_safe
    # Candle anatomy.
    feats["candle_body"] = (close - open_) / atr_safe
    feats["upper_wick"] = (high - np.maximum(close, open_)) / atr_safe
    feats["lower_wick"] = (np.minimum(close, open_) - low) / atr_safe
    feats["range_atr"] = (high - low) / atr_safe
    # Volume anomalies.
    for window in (20, 100):
        vol_mean = volume.rolling(window, min_periods=window).mean()
        vol_std = volume.rolling(window, min_periods=window).std()
        feats[f"volume_z_{window}"] = (volume - vol_mean) / vol_std.replace(0.0, np.nan)
    # Position inside recent ranges and distance to swing extremes.
    for window in (24, 72, 144):
        hh = high.rolling(window, min_periods=window).max()
        ll = low.rolling(window, min_periods=window).min()
        feats[f"range_pos_{window}"] = (close - ll) / (hh - ll).replace(0.0, np.nan)
        feats[f"dist_high_{window}_atr"] = (hh - close) / atr_safe
        feats[f"dist_low_{window}_atr"] = (close - ll) / atr_safe
    # Consecutive same-direction closes (signed).
    sign = np.sign(close.diff()).fillna(0.0)
    groups = (sign != sign.shift()).cumsum()
    feats["streak"] = sign * (sign.groupby(groups).cumcount() + 1)
    # Intraday / weekly seasonality.
    ts = pd.to_datetime(out["time"], utc=True)
    hour = ts.dt.hour + ts.dt.minute / 60.0
    feats["hour_sin"] = np.sin(2.0 * np.pi * hour / 24.0)
    feats["hour_cos"] = np.cos(2.0 * np.pi * hour / 24.0)
    feats["dow_sin"] = np.sin(2.0 * np.pi * ts.dt.dayofweek / 7.0)
    feats["dow_cos"] = np.cos(2.0 * np.pi * ts.dt.dayofweek / 7.0)
    # Short raw history of the last six candle returns (ATR units).
    for lag in range(1, 7):
        feats[f"ret_lag{lag}_atr"] = (close.shift(lag - 1) - close.shift(lag)) / atr_safe
    return pd.concat([out, pd.DataFrame(feats, index=out.index)], axis=1)


featured = add_opportunity_features(bars, atr_period=int(CONFIG["atr_period"]))
FEATURE_COLS: Tuple[str, ...] = tuple(
    c for c in featured.columns if c not in OHLCV_COLS and c != "atr"
)
print(len(FEATURE_COLS), "features")
null_rates = featured[list(FEATURE_COLS)].isna().mean()
print("max feature null rate", round(float(null_rates.max()), 4), "at", null_rates.idxmax())


46 features
max feature null rate 0.0047 at ema_slope_200_atr


## 07 Opportunity labels


In [ ]:
# CELL: LABELS
# Opportunity labels on a closed-bar series. Path is bars t+1 .. t+horizon only.
# Vectorised over bars; semantics are identical to walking each path bar by bar.

import numpy as np
import pandas as pd

_EPS = 1e-9
HORIZON_BARS_DEFAULT = 24
ATR_PERIOD_DEFAULT = 14
TP_ATR_MULT_DEFAULT = 4.0
SL_ATR_MULT_DEFAULT = 2.0

TP_FIRST = 0
SL_FIRST = 1
NEITHER = 2
AMBIGUOUS = 3

LABEL_LONG = 0
LABEL_SHORT = 1
LABEL_NO_TRADE = 2
LABEL_UNLABELED = -1
LABEL_NAMES = ("LONG", "SHORT", "NO_TRADE")


def tp_sl_from_path(
    fwd_high: np.ndarray,
    fwd_low: np.ndarray,
    entry: float,
    atr: float,
    tp_mult: float,
    sl_mult: float,
    side: str,
) -> int:
    """Walk future highs/lows. Same-bar TP and SL is AMBIGUOUS, not a win."""
    high = np.asarray(fwd_high, dtype=np.float64).ravel()
    low = np.asarray(fwd_low, dtype=np.float64).ravel()
    if high.size == 0 or high.size != low.size:
        return int(NEITHER)
    denom = max(float(atr), _EPS)
    entry_f = float(entry)
    if str(side).strip().upper() == "SHORT":
        tp_level = entry_f - float(tp_mult) * denom
        sl_level = entry_f + float(sl_mult) * denom
        tp_hit = low <= tp_level
        sl_hit = high >= sl_level
    else:
        tp_level = entry_f + float(tp_mult) * denom
        sl_level = entry_f - float(sl_mult) * denom
        tp_hit = high >= tp_level
        sl_hit = low <= sl_level
    for j in range(high.size):
        hit_tp = bool(tp_hit[j])
        hit_sl = bool(sl_hit[j])
        if hit_tp and hit_sl:
            return int(AMBIGUOUS)
        if hit_tp:
            return int(TP_FIRST)
        if hit_sl:
            return int(SL_FIRST)
    return int(NEITHER)


def _forward_stack(values: np.ndarray, k: int) -> np.ndarray:
    """(k, n) array where row j-1 holds values shifted back by j bars (NaN padded)."""
    n = values.shape[0]
    out = np.full((k, n), np.nan, dtype=np.float64)
    for j in range(1, k + 1):
        if n - j > 0:
            out[j - 1, : n - j] = values[j:]
    return out


def _side_codes_vectorised(
    fwd_high: np.ndarray,
    fwd_low: np.ndarray,
    entry: np.ndarray,
    atr: np.ndarray,
    tp_mult: float,
    sl_mult: float,
    side: str,
) -> np.ndarray:
    """Vectorised twin of ``tp_sl_from_path`` for every bar at once."""
    denom = np.maximum(atr, _EPS)
    if str(side).strip().upper() == "SHORT":
        tp_level = entry - float(tp_mult) * denom
        sl_level = entry + float(sl_mult) * denom
        tp_hits = fwd_low <= tp_level[None, :]
        sl_hits = fwd_high >= sl_level[None, :]
    else:
        tp_level = entry + float(tp_mult) * denom
        sl_level = entry - float(sl_mult) * denom
        tp_hits = fwd_high >= tp_level[None, :]
        sl_hits = fwd_low <= sl_level[None, :]
    any_hit = tp_hits | sl_hits
    has_hit = any_hit.any(axis=0)
    first = np.argmax(any_hit, axis=0)
    cols = np.arange(entry.shape[0])
    tp_first = tp_hits[first, cols]
    sl_first = sl_hits[first, cols]
    codes = np.where(
        ~has_hit,
        NEITHER,
        np.where(tp_first & sl_first, AMBIGUOUS, np.where(tp_first, TP_FIRST, SL_FIRST)),
    )
    return codes.astype(np.int64)


def _causal_atr(df: pd.DataFrame, period: int = ATR_PERIOD_DEFAULT) -> np.ndarray:
    if "atr" in df.columns:
        atr = df["atr"].to_numpy(dtype=np.float64)
        if np.isfinite(atr).any():
            return atr
    close = df["close"].to_numpy(dtype=np.float64)
    high = df["high"].to_numpy(dtype=np.float64)
    low = df["low"].to_numpy(dtype=np.float64)
    prev = np.concatenate([[close[0]], close[:-1]])
    tr = np.maximum(high - low, np.maximum(np.abs(high - prev), np.abs(low - prev)))
    return pd.Series(tr).rolling(int(period), min_periods=int(period)).mean().to_numpy(
        dtype=np.float64
    )


def compute_opportunity_labels(
    df: pd.DataFrame,
    *,
    tp_mult: float = TP_ATR_MULT_DEFAULT,
    sl_mult: float = SL_ATR_MULT_DEFAULT,
    horizon_bars: int = HORIZON_BARS_DEFAULT,
    atr_period: int = ATR_PERIOD_DEFAULT,
) -> pd.DataFrame:
    """Label each closed bar. Last ``horizon_bars`` rows stay unlabeled.

    Realised P&L columns (``long_pnl_pct`` / ``short_pnl_pct``) are gross of
    cost: +TP distance on TP_FIRST, −SL distance on SL_FIRST **and** AMBIGUOUS
    (a same-bar double touch is scored as a stop), timeout return on NEITHER.
    """
    out = df.copy().reset_index(drop=True)
    n = len(out)
    k = int(horizon_bars)
    close = out["close"].to_numpy(dtype=np.float64)
    high = out["high"].to_numpy(dtype=np.float64)
    low = out["low"].to_numpy(dtype=np.float64)
    atr = np.asarray(_causal_atr(out, period=int(atr_period)), dtype=np.float64)
    if atr.shape[0] != n:
        atr = np.resize(atr, n)

    fwd_high = _forward_stack(high, k)
    fwd_low = _forward_stack(low, k)
    fwd_close_end = np.full(n, np.nan, dtype=np.float64)
    if n - k > 0:
        fwd_close_end[: n - k] = close[k:]

    index = np.arange(n)
    valid = (
        (index < n - k)
        & np.isfinite(atr)
        & (atr > _EPS)
        & np.isfinite(close)
        & (np.abs(close) > _EPS)
    )
    atr_safe = np.where(valid, atr, np.nan)
    entry_safe = np.where(valid, close, np.nan)

    l_code = _side_codes_vectorised(fwd_high, fwd_low, entry_safe, atr_safe, tp_mult, sl_mult, "LONG")
    s_code = _side_codes_vectorised(fwd_high, fwd_low, entry_safe, atr_safe, tp_mult, sl_mult, "SHORT")
    l_win = l_code == TP_FIRST
    s_win = s_code == TP_FIRST
    label = np.where(l_win & ~s_win, LABEL_LONG, np.where(s_win & ~l_win, LABEL_SHORT, LABEL_NO_TRADE))

    with np.errstate(invalid="ignore", divide="ignore"):
        max_high = np.nanmax(np.where(np.isfinite(fwd_high), fwd_high, -np.inf), axis=0)
        min_low = np.nanmin(np.where(np.isfinite(fwd_low), fwd_low, np.inf), axis=0)
        upside = np.maximum((max_high - entry_safe) / atr_safe, 0.0)
        downside = np.maximum((entry_safe - min_low) / atr_safe, 0.0)
        timeout_ret = (fwd_close_end - entry_safe) / entry_safe
        tp_pct = float(tp_mult) * atr_safe / entry_safe
        sl_pct = float(sl_mult) * atr_safe / entry_safe
        long_pnl = np.where(
            l_code == TP_FIRST, tp_pct, np.where(l_code == NEITHER, timeout_ret, -sl_pct)
        )
        short_pnl = np.where(
            s_code == TP_FIRST, tp_pct, np.where(s_code == NEITHER, -timeout_ret, -sl_pct)
        )

    def _masked(values: np.ndarray) -> np.ndarray:
        return np.where(valid, values, np.nan)

    out["atr"] = atr
    out["trade_label"] = np.where(valid, label, LABEL_UNLABELED).astype(np.int64)
    out["long_result"] = _masked(l_win.astype(np.float64))
    out["short_result"] = _masked(s_win.astype(np.float64))
    out["long_code"] = _masked(l_code.astype(np.float64))
    out["short_code"] = _masked(s_code.astype(np.float64))
    out["upside"] = _masked(upside)
    out["downside"] = _masked(downside)
    out["timeout_ret"] = _masked(timeout_ret)
    out["tp_pct"] = _masked(tp_pct)
    out["sl_pct"] = _masked(sl_pct)
    out["long_pnl_pct"] = _masked(long_pnl)
    out["short_pnl_pct"] = _masked(short_pnl)
    names = np.array(["UNLABELED"] * n, dtype=object)
    names[out["trade_label"].to_numpy() == LABEL_LONG] = "LONG"
    names[out["trade_label"].to_numpy() == LABEL_SHORT] = "SHORT"
    names[out["trade_label"].to_numpy() == LABEL_NO_TRADE] = "NO_TRADE"
    out["trade_name"] = names
    return out


## 08 Apply labels


In [ ]:
# CELL: APPLY_LABELS
labeled = compute_opportunity_labels(
    featured,
    tp_mult=float(CONFIG["tp_atr_mult"]),
    sl_mult=float(CONFIG["sl_atr_mult"]),
    horizon_bars=int(CONFIG["horizon_bars"]),
    atr_period=int(CONFIG["atr_period"]),
)
valid = labeled["trade_label"].to_numpy() >= 0
print(f"labeled rows={int(valid.sum())} / {len(labeled)}")
print(labeled.loc[valid, "trade_name"].value_counts(normalize=True).round(4))
print(
    "median TP distance",
    f"{labeled.loc[valid, 'tp_pct'].median():.4%}",
    "median SL distance",
    f"{labeled.loc[valid, 'sl_pct'].median():.4%}",
    "round trip cost",
    f"{float(CONFIG['round_trip_cost_pct']):.4%}",
)


labeled rows=43163 / 43200
trade_name
NO_TRADE    0.5985
SHORT       0.2022
LONG        0.1993
Name: proportion, dtype: float64
median TP distance 1.6166% median SL distance 0.8083% round trip cost 0.1600%


## 09 Train-period gate


In [ ]:
# CELL: GATE
def train_period_mask(n: int, train_fraction: float) -> np.ndarray:
    cut = int(n * float(train_fraction))
    mask = np.zeros(n, dtype=bool)
    mask[:cut] = True
    return mask


def _path_code_counts(codes: np.ndarray, usable: np.ndarray) -> Dict[str, int]:
    finite = usable & np.isfinite(codes)
    return {
        "TP_FIRST": int(np.sum(finite & (codes == TP_FIRST))),
        "SL_FIRST": int(np.sum(finite & (codes == SL_FIRST))),
        "NEITHER": int(np.sum(finite & (codes == NEITHER))),
        "AMBIGUOUS": int(np.sum(finite & (codes == AMBIGUOUS))),
    }


def summarize_label_gate(
    df: pd.DataFrame,
    *,
    train_fraction: float,
    round_trip_cost_pct: float,
    minority_rate: float,
    ambiguous_win_max: float,
    min_tp_cost_ratio: float = 4.0,
) -> Dict[str, Any]:
    """Decide whether the bracket is worth training on, using train rows only.

    Besides class mix and oracle exclusive expectancy, the gate checks the
    bracket economics: the median take-profit distance must be at least
    ``min_tp_cost_ratio`` times the round-trip cost, and the implied
    break-even win rate ``(sl + cost) / (tp + sl)`` is reported.
    """
    n = len(df)
    in_train = train_period_mask(n, train_fraction)
    labels = df["trade_label"].to_numpy(dtype=np.int64)
    usable = in_train & (labels >= 0)
    n_use = int(usable.sum())
    if n_use == 0:
        raise RuntimeError("Train-period gate has no labeled rows")
    mix = {}
    mix_ok = True
    for cid, name in enumerate(LABEL_NAMES):
        rate = float(np.mean(labels[usable] == cid))
        mix[name] = rate
        if rate < float(minority_rate):
            mix_ok = False
    long_code = df["long_code"].to_numpy(dtype=np.float64)
    short_code = df["short_code"].to_numpy(dtype=np.float64)
    cost = float(round_trip_cost_pct)
    long_pnl = df["long_pnl_pct"].to_numpy(dtype=np.float64) - cost
    short_pnl = df["short_pnl_pct"].to_numpy(dtype=np.float64) - cost

    always_long_mask = usable & np.isfinite(long_pnl)
    always_short_mask = usable & np.isfinite(short_pnl)
    always_in_long = (
        float(np.mean(long_pnl[always_long_mask])) if always_long_mask.any() else float("nan")
    )
    always_in_short = (
        float(np.mean(short_pnl[always_short_mask])) if always_short_mask.any() else float("nan")
    )

    excl_long_mask = usable & (labels == LABEL_LONG) & np.isfinite(long_pnl)
    excl_short_mask = usable & (labels == LABEL_SHORT) & np.isfinite(short_pnl)
    exclusive_long = (
        float(np.mean(long_pnl[excl_long_mask])) if excl_long_mask.any() else float("nan")
    )
    exclusive_short = (
        float(np.mean(short_pnl[excl_short_mask])) if excl_short_mask.any() else float("nan")
    )

    def _ambig_share(codes: np.ndarray) -> float:
        finite = usable & np.isfinite(codes)
        tp = finite & (codes == TP_FIRST)
        amb = finite & (codes == AMBIGUOUS)
        denom = int(tp.sum() + amb.sum())
        if denom == 0:
            return 0.0
        return float(amb.sum()) / float(denom)

    long_ambig = _ambig_share(long_code)
    short_ambig = _ambig_share(short_code)

    if "tp_pct" in df.columns and "sl_pct" in df.columns:
        tp_pct = df["tp_pct"].to_numpy(dtype=np.float64)
        sl_pct = df["sl_pct"].to_numpy(dtype=np.float64)
    else:
        atr_pct = df["atr"].to_numpy(dtype=np.float64) / df["close"].to_numpy(dtype=np.float64)
        tp_pct = float(CONFIG["tp_atr_mult"]) * atr_pct
        sl_pct = float(CONFIG["sl_atr_mult"]) * atr_pct
    econ_mask = usable & np.isfinite(tp_pct) & np.isfinite(sl_pct)
    median_tp = float(np.median(tp_pct[econ_mask])) if econ_mask.any() else float("nan")
    median_sl = float(np.median(sl_pct[econ_mask])) if econ_mask.any() else float("nan")
    tp_cost_ratio = median_tp / cost if cost > 0 and np.isfinite(median_tp) else float("inf")
    breakeven_win_rate = (
        (median_sl + cost) / (median_tp + median_sl)
        if np.isfinite(median_tp) and np.isfinite(median_sl) and (median_tp + median_sl) > 0
        else float("nan")
    )
    long_tp_rate = float(np.mean(long_code[usable & np.isfinite(long_code)] == TP_FIRST))
    short_tp_rate = float(np.mean(short_code[usable & np.isfinite(short_code)] == TP_FIRST))

    reasons: List[str] = []
    if not mix_ok:
        reasons.append("class_mix_below_floor")
    if not (np.isfinite(exclusive_long) and exclusive_long > 0.0):
        reasons.append("exclusive_long_expectancy_not_positive")
    if not (np.isfinite(exclusive_short) and exclusive_short > 0.0):
        reasons.append("exclusive_short_expectancy_not_positive")
    if long_ambig >= float(ambiguous_win_max):
        reasons.append("long_ambiguous_majority")
    if short_ambig >= float(ambiguous_win_max):
        reasons.append("short_ambiguous_majority")
    if tp_cost_ratio < float(min_tp_cost_ratio):
        reasons.append("tp_distance_below_cost_ratio")
    passed = len(reasons) == 0
    return {
        "passed": passed,
        "n_train_labeled": n_use,
        "class_mix": mix,
        "minority_rate": float(minority_rate),
        "exclusive_long_expectancy_pct": exclusive_long,
        "exclusive_short_expectancy_pct": exclusive_short,
        "always_in_long_expectancy_pct": always_in_long,
        "always_in_short_expectancy_pct": always_in_short,
        "round_trip_cost_pct": cost,
        "median_tp_pct": median_tp,
        "median_sl_pct": median_sl,
        "tp_cost_ratio": tp_cost_ratio,
        "min_tp_cost_ratio": float(min_tp_cost_ratio),
        "breakeven_win_rate": breakeven_win_rate,
        "base_long_tp_rate": long_tp_rate,
        "base_short_tp_rate": short_tp_rate,
        "long_ambiguous_share": long_ambig,
        "short_ambiguous_share": short_ambig,
        "long_path_codes": _path_code_counts(long_code, usable),
        "short_path_codes": _path_code_counts(short_code, usable),
        "tp_atr_mult": float(CONFIG["tp_atr_mult"]),
        "sl_atr_mult": float(CONFIG["sl_atr_mult"]),
        "reasons": reasons,
    }


EXPORT_DIR = Path(str(CONFIG["export_dir"]))
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
label_report = summarize_label_gate(
    labeled,
    train_fraction=float(CONFIG["train_fraction"]),
    round_trip_cost_pct=float(CONFIG["round_trip_cost_pct"]),
    minority_rate=float(CONFIG["minority_rate"]),
    ambiguous_win_max=float(CONFIG["ambiguous_win_max"]),
    min_tp_cost_ratio=float(CONFIG["min_tp_cost_ratio"]),
)
report_path = EXPORT_DIR / "label_report.json"
report_path.write_text(json.dumps(label_report, indent=2), encoding="utf-8")
print(json.dumps(label_report, indent=2))
print("wrote", report_path)
if not label_report["passed"]:
    raise RuntimeError(
        "Train-period label gate failed: "
        f"{label_report['reasons']}. Change CONFIG tp/sl ATR multiples or horizon and "
        "re-run from the top. Do not fit models."
    )
print(
    f"Gate passed. Break-even win rate {label_report['breakeven_win_rate']:.1%} vs base "
    f"TP rates long {label_report['base_long_tp_rate']:.1%} / short "
    f"{label_report['base_short_tp_rate']:.1%}. Training may start."
)


{
  "passed": true,
  "n_train_labeled": 30226,
  "class_mix": {
    "LONG": 0.19790908489380005,
    "SHORT": 0.19585787070733807,
    "NO_TRADE": 0.6062330443988619
  },
  "minority_rate": 0.15,
  "exclusive_long_expectancy_pct": 0.014104124587297387,
  "exclusive_short_expectancy_pct": 0.01487245416371671,
  "always_in_long_expectancy_pct": -0.001310670653988737,
  "always_in_short_expectancy_pct": -0.001857664666695946,
  "round_trip_cost_pct": 0.0016,
  "median_tp_pct": 0.016473414606161106,
  "median_sl_pct": 0.008236707303080553,
  "tp_cost_ratio": 10.295884128850691,
  "min_tp_cost_ratio": 4.0,
  "breakeven_win_rate": 0.3980841267886095,
  "base_long_tp_rate": 0.19790908489380005,
  "base_short_tp_rate": 0.19585787070733807,
  "long_ambiguous_share": 0.0016688918558077437,
  "short_ambiguous_share": 0.0015179625569235959,
  "long_path_codes": {
    "TP_FIRST": 5982,
    "SL_FIRST": 14709,
    "NEITHER": 9525,
    "AMBIGUOUS": 10
  },
  "short_path_codes": {
    "TP_FIRST": 5920

## 10 Temporal split


In [ ]:
# CELL: SPLIT
def purged_index_splits(
    n: int,
    *,
    train_fraction: float,
    val_fraction: float,
    purge_bars: int,
) -> Dict[str, np.ndarray]:
    """Time-ordered split. Purge ``purge_bars`` on each side of every cut."""
    n = int(n)
    purge = int(purge_bars)
    train_end = int(n * float(train_fraction))
    val_len = int(n * float(val_fraction))
    train_idx = np.arange(0, max(train_end - purge, 0), dtype=np.int64)
    val_start = min(train_end + purge, n)
    val_end = min(val_start + val_len, n)
    val_idx = np.arange(val_start, max(val_end - purge, val_start), dtype=np.int64)
    test_start = min(val_end + purge, n)
    test_idx = np.arange(test_start, n, dtype=np.int64)
    return {"train": train_idx, "val": val_idx, "test": test_idx}


def development_walk_forward(
    n_dev: int,
    *,
    folds: int = 4,
    purge_bars: int,
) -> List[Tuple[np.ndarray, np.ndarray]]:
    """Expanding-window folds inside the development prefix only."""
    folds = max(2, int(folds))
    purge = int(purge_bars)
    edges = [int(n_dev * (i + 1) / folds) for i in range(folds)]
    out: List[Tuple[np.ndarray, np.ndarray]] = []
    for i in range(1, folds):
        train_end = edges[i - 1]
        val_end = edges[i]
        tr = np.arange(0, max(train_end - purge, 0), dtype=np.int64)
        va = np.arange(min(train_end + purge, n_dev), val_end, dtype=np.int64)
        if len(tr) == 0 or len(va) == 0:
            continue
        out.append((tr, va))
    return out


def inner_calibration_split(
    train_idx: np.ndarray, *, calib_fraction: float, purge_bars: int
) -> Tuple[np.ndarray, np.ndarray]:
    """Reserve the purged tail of a training window for threshold selection."""
    n = len(train_idx)
    cut = int(n * (1.0 - float(calib_fraction)))
    fit_idx = train_idx[: max(cut - int(purge_bars), 0)]
    calib_idx = train_idx[cut:]
    return fit_idx, calib_idx


splits = purged_index_splits(
    len(labeled),
    train_fraction=float(CONFIG["train_fraction"]),
    val_fraction=float(CONFIG["val_fraction"]),
    purge_bars=int(CONFIG["purge_bars"]),
)
for key, idx in splits.items():
    print(key, "n", len(idx), "range", int(idx[0]) if len(idx) else None, int(idx[-1]) if len(idx) else None)
dev_end = int(splits["test"][0]) if len(splits["test"]) else len(labeled)
wf_folds = development_walk_forward(
    dev_end, folds=int(CONFIG["walk_forward_folds"]), purge_bars=int(CONFIG["purge_bars"])
)
print("walk-forward folds", len(wf_folds), "dev_end", dev_end)


train n 30214 range 0 30213
val n 6455 range 30264 36718
test n 6431 range 36769 43199
walk-forward folds 3 dev_end 36769


## 11 Scaler and datasets


In [ ]:
# CELL: DATASET
def build_rows(
    frame: pd.DataFrame, *, feature_cols: Sequence[str]
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """One tabular row per labeled bar with finite features and finite path targets."""
    feats = frame.loc[:, list(feature_cols)].to_numpy(dtype=np.float64)
    y_cls = frame["trade_label"].to_numpy(dtype=np.int64)
    y_reg = frame.loc[:, ["upside", "downside"]].to_numpy(dtype=np.float64)
    ok = (y_cls >= 0) & np.isfinite(feats).all(axis=1) & np.isfinite(y_reg).all(axis=1)
    rows = np.flatnonzero(ok).astype(np.int64)
    return (
        feats[rows].astype(np.float32),
        y_cls[rows],
        y_reg[rows].astype(np.float32),
        rows,
    )


all_x, all_y, all_r, all_rows = build_rows(labeled, feature_cols=FEATURE_COLS)
print(
    "rows",
    all_x.shape,
    "class mix",
    {n: round(float(np.mean(all_y == i)), 4) for i, n in enumerate(LABEL_NAMES)},
)


def rows_in(split_idx: np.ndarray) -> np.ndarray:
    """Boolean selector over ``all_rows`` for a set of frame indices."""
    allowed = np.zeros(len(labeled), dtype=bool)
    allowed[np.asarray(split_idx, dtype=np.int64)] = True
    return allowed[all_rows]


MIN_SIGNAL_TP_PCT = float(CONFIG["min_tp_cost_ratio"]) * float(CONFIG["round_trip_cost_pct"])


def signal_eligible(rows: np.ndarray) -> np.ndarray:
    """Quiet-market guard: a candle may carry a signal only if its TP distance pays for cost."""
    tp_pct = labeled["tp_pct"].to_numpy(dtype=np.float64)[rows]
    return np.isfinite(tp_pct) & (tp_pct >= MIN_SIGNAL_TP_PCT)


print(
    f"signal eligibility: TP distance >= {MIN_SIGNAL_TP_PCT:.4%} -> "
    f"{float(signal_eligible(all_rows).mean()):.1%} of labeled candles"
)


def fit_scaler(x: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    mean = x.mean(axis=0, keepdims=True)
    std = x.std(axis=0, keepdims=True)
    std = np.where(std < 1e-6, 1.0, std)
    return mean.astype(np.float32), std.astype(np.float32)


def scale_x(x: np.ndarray, mean: np.ndarray, std: np.ndarray) -> np.ndarray:
    return ((x - mean) / std).astype(np.float32)


sel_train = rows_in(splits["train"])
sel_val = rows_in(splits["val"])
sel_test = rows_in(splits["test"])
feat_mean, feat_std = fit_scaler(all_x[sel_train])
reg_mean = all_r[sel_train].mean(axis=0, keepdims=True)
reg_std = all_r[sel_train].std(axis=0, keepdims=True)
reg_std = np.where(reg_std < 1e-6, 1.0, reg_std)


def scale_r(r: np.ndarray) -> np.ndarray:
    return ((r - reg_mean) / reg_std).astype(np.float32)


X_train = scale_x(all_x[sel_train], feat_mean, feat_std)
X_val = scale_x(all_x[sel_val], feat_mean, feat_std)
X_test = scale_x(all_x[sel_test], feat_mean, feat_std)
y_train, y_val, y_test = all_y[sel_train], all_y[sel_val], all_y[sel_test]
r_train, r_val, r_test = all_r[sel_train], all_r[sel_val], all_r[sel_test]
R_train, R_val, R_test = scale_r(r_train), scale_r(r_val), scale_r(r_test)
print("train", X_train.shape, "val", X_val.shape, "test", X_test.shape)


rows (42971, 46) class mix {'LONG': 0.1996, 'SHORT': 0.2022, 'NO_TRADE': 0.5983}
signal eligibility: TP distance >= 0.6400% -> 95.3% of labeled candles
train (30009, 46) val (6455, 46) test (6407, 46)


## 12 Model families (LR, RF, XGB, LGBM, CatBoost)


In [ ]:
# CELL: MODEL
def make_family(name: str, seed: int) -> Dict[str, Any]:
    """Return unfitted classifier + two regressors for one family.

    No class re-weighting anywhere: the decision rule thresholds calibrated-ish
    probabilities, and ``balanced`` weights distort them.
    """
    est = int(CONFIG["n_estimators"])
    depth = int(CONFIG["max_depth"])
    lr = float(CONFIG["learning_rate"])
    leaf = int(CONFIG["min_leaf"])
    threads = int(CONFIG["threads_per_model"])
    if name == "LR":
        clf = LogisticRegression(max_iter=3000, C=0.1, solver="lbfgs")
        up = Ridge(alpha=10.0)
        dn = Ridge(alpha=10.0)
    elif name == "RF":
        rf_kwargs = dict(
            n_estimators=est,
            max_depth=depth + 2,
            min_samples_leaf=leaf,
            max_features="sqrt",
            n_jobs=threads,
        )
        clf = RandomForestClassifier(random_state=seed, **rf_kwargs)
        up = RandomForestRegressor(random_state=seed, **rf_kwargs)
        dn = RandomForestRegressor(random_state=seed + 1, **rf_kwargs)
    elif name == "XGB":
        xgb_kwargs = dict(
            n_estimators=est,
            max_depth=depth,
            learning_rate=lr,
            min_child_weight=leaf,
            subsample=0.8,
            colsample_bytree=0.7,
            reg_lambda=5.0,
            tree_method="hist",
            n_jobs=threads,
            verbosity=0,
        )
        clf = XGBClassifier(
            objective="multi:softprob", eval_metric="mlogloss", random_state=seed, **xgb_kwargs
        )
        up = XGBRegressor(random_state=seed, **xgb_kwargs)
        dn = XGBRegressor(random_state=seed + 1, **xgb_kwargs)
    elif name == "LGBM":
        lgb_kwargs = dict(
            n_estimators=est,
            num_leaves=31,
            max_depth=-1,
            learning_rate=lr,
            min_child_samples=leaf,
            subsample=0.8,
            subsample_freq=1,
            colsample_bytree=0.7,
            reg_lambda=5.0,
            n_jobs=threads,
            verbosity=-1,
        )
        clf = LGBMClassifier(random_state=seed, **lgb_kwargs)
        up = LGBMRegressor(random_state=seed, **lgb_kwargs)
        dn = LGBMRegressor(random_state=seed + 1, **lgb_kwargs)
    elif name == "CatBoost":
        cb_kwargs = dict(
            iterations=est,
            depth=depth,
            learning_rate=lr,
            l2_leaf_reg=10.0,
            verbose=False,
            thread_count=threads,
            allow_writing_files=False,
        )
        clf = CatBoostClassifier(loss_function="MultiClass", random_seed=seed, **cb_kwargs)
        up = CatBoostRegressor(random_seed=seed, **cb_kwargs)
        dn = CatBoostRegressor(random_seed=seed + 1, **cb_kwargs)
    else:
        raise ValueError(name)
    return {"name": name, "clf": clf, "up": up, "dn": dn}


print("families", CONFIG["ensemble"], "+", ENSEMBLE_NAME)


families ['XGB'] + ENSEMBLE


## 13 Walk-forward bake-off, threshold selection, final fit


In [ ]:
# CELL: TRAIN
if not CONFIG.get("run_baseline_train", True):
    print("Skipping baseline TRAIN")
else:
    def confusion(y_true: np.ndarray, y_pred: np.ndarray, n_classes: int = 3) -> np.ndarray:
        mat = np.zeros((n_classes, n_classes), dtype=np.int64)
        for t, p in zip(y_true.tolist(), y_pred.tolist()):
            if 0 <= int(t) < n_classes and 0 <= int(p) < n_classes:
                mat[int(t), int(p)] += 1
        return mat


    def class_precision(mat: np.ndarray, cls: int) -> float:
        col = mat[:, cls]
        denom = float(col.sum())
        if denom <= 0:
            return float("nan")
        return float(mat[cls, cls]) / denom


    def balanced_accuracy(mat: np.ndarray) -> float:
        recalls = []
        for i in range(mat.shape[0]):
            denom = float(mat[i].sum())
            recalls.append(float(mat[i, i]) / denom if denom else float("nan"))
        finite = [r for r in recalls if np.isfinite(r)]
        return float(np.mean(finite)) if finite else float("nan")


    def corr(a: np.ndarray, b: np.ndarray) -> float:
        if a.size < 3:
            return float("nan")
        if float(np.std(a)) < 1e-12 or float(np.std(b)) < 1e-12:
            return float("nan")
        return float(np.corrcoef(a, b)[0, 1])


    def as_cls(arr: np.ndarray) -> np.ndarray:
        return np.asarray(arr).reshape(-1).astype(np.int64)


    def as_1d(arr: np.ndarray) -> np.ndarray:
        return np.asarray(arr, dtype=np.float64).reshape(-1)


    def proba_of(clf: Any, X: np.ndarray) -> np.ndarray:
        """(n, 3) class probabilities in LONG/SHORT/NO_TRADE order regardless of estimator."""
        p = np.asarray(clf.predict_proba(X), dtype=np.float64)
        if p.ndim == 1:
            p = p.reshape(len(X), -1)
        classes = getattr(clf, "classes_", None)
        if classes is None:
            return p
        full = np.zeros((p.shape[0], len(LABEL_NAMES)), dtype=np.float64)
        for j, c in enumerate(np.asarray(classes).reshape(-1).astype(int)):
            full[:, c] = p[:, j]
        return full


    def classifier_report(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, Any]:
        cm = confusion(y_true, y_pred)
        return {
            "precision_LONG": class_precision(cm, LABEL_LONG),
            "precision_SHORT": class_precision(cm, LABEL_SHORT),
            "n_LONG": float(np.sum(y_pred == LABEL_LONG)),
            "n_SHORT": float(np.sum(y_pred == LABEL_SHORT)),
            "balanced_accuracy": balanced_accuracy(cm),
            "confusion": cm.tolist(),
        }


    _PNL_COL = {LABEL_LONG: "long_pnl_pct", LABEL_SHORT: "short_pnl_pct"}
    _SIDE_NAME = {LABEL_LONG: "LONG", LABEL_SHORT: "SHORT"}


    def net_pnl(cls: int, rows: np.ndarray, cost: Optional[float] = None) -> np.ndarray:
        """Net-of-cost realised P&L (fraction of price) for taking ``cls`` on ``rows``."""
        c = float(CONFIG["round_trip_cost_pct"]) if cost is None else float(cost)
        return labeled[_PNL_COL[int(cls)]].to_numpy(dtype=np.float64)[rows] - c


    def signal_expectancy(
        pred_cls: np.ndarray, rows: np.ndarray, cost: Optional[float] = None
    ) -> Dict[str, float]:
        """Per-side expectancy, t-stat, profit factor and total for the predicted signals."""
        out: Dict[str, float] = {}
        for cls, name in _SIDE_NAME.items():
            take = pred_cls == cls
            pnl = net_pnl(cls, rows[take], cost) if take.any() else np.zeros(0)
            pnl = pnl[np.isfinite(pnl)]
            n = int(pnl.size)
            out[f"{name}_n"] = float(n)
            out[f"{name}_expectancy"] = float(pnl.mean()) if n else float("nan")
            std = float(pnl.std(ddof=1)) if n > 1 else float("nan")
            out[f"{name}_tstat"] = (
                float(pnl.mean() / (std / math.sqrt(n))) if n > 1 and std > 0 else float("nan")
            )
            wins = float(pnl[pnl > 0].sum())
            losses = float(-pnl[pnl < 0].sum())
            out[f"{name}_profit_factor"] = (
                wins / losses if losses > 0 else (float("inf") if wins > 0 else float("nan"))
            )
            out[f"{name}_total_pct"] = float(pnl.sum())
        return out


    def expectancy_rank_score(exp_report: Dict[str, float]) -> float:
        """Mean expectancy over sides with >= ``min_signals`` trades; -inf when none qualify."""
        min_n = int(CONFIG["min_signals"])
        parts: List[float] = []
        for name in ("LONG", "SHORT"):
            n = float(exp_report.get(f"{name}_n", 0.0))
            e = float(exp_report.get(f"{name}_expectancy", float("nan")))
            if n >= min_n and np.isfinite(e):
                parts.append(e)
        return float(np.mean(parts)) if parts else float("-inf")


    def choose_thresholds(proba: np.ndarray, rows: np.ndarray) -> Dict[str, Optional[float]]:
        """Per side, the probability threshold that maximises calibration expectancy.

        Candidates are quantiles of the side's probability on the calibration rows.
        A side with no candidate reaching ``min_signals`` trades is disabled (None).
        """
        min_n = int(CONFIG["min_signals"])
        eligible = signal_eligible(rows)
        out: Dict[str, Optional[float]] = {}
        for cls, name in _SIDE_NAME.items():
            other = LABEL_SHORT if cls == LABEL_LONG else LABEL_LONG
            p_side = proba[:, cls]
            p_other = proba[:, other]
            pnl_all = net_pnl(cls, rows)
            best_thr: Optional[float] = None
            best_e = float("-inf")
            if not eligible.any():
                out[name] = None
                continue
            for q in CONFIG["signal_quantiles"]:
                thr = float(np.quantile(p_side[eligible], float(q)))
                take = eligible & (p_side >= thr) & (p_side > p_other) & np.isfinite(pnl_all)
                n = int(take.sum())
                if n < min_n:
                    continue
                e = float(pnl_all[take].mean())
                if e > best_e:
                    best_thr, best_e = thr, e
            out[name] = best_thr
        return out


    def apply_thresholds(
        proba: np.ndarray, thr: Dict[str, Optional[float]], eligible: Optional[np.ndarray] = None
    ) -> np.ndarray:
        """Live decision rule.

        Abstain unless the candle is eligible (TP distance pays for cost), P(side)
        clears the side's threshold and beats the other side.
        """
        pred = np.full(proba.shape[0], LABEL_NO_TRADE, dtype=np.int64)
        ok = np.ones(proba.shape[0], dtype=bool) if eligible is None else np.asarray(eligible, dtype=bool)
        p_long, p_short = proba[:, LABEL_LONG], proba[:, LABEL_SHORT]
        if thr.get("LONG") is not None:
            pred[ok & (p_long >= float(thr["LONG"])) & (p_long > p_short)] = LABEL_LONG
        if thr.get("SHORT") is not None:
            pred[ok & (p_short >= float(thr["SHORT"])) & (p_short > p_long)] = LABEL_SHORT
        return pred


    def score_probas(
        p_calib: np.ndarray,
        rows_calib: np.ndarray,
        p_eval: np.ndarray,
        rows_eval: np.ndarray,
        y_eval: np.ndarray,
    ) -> Dict[str, Any]:
        thr = choose_thresholds(p_calib, rows_calib)
        pred = apply_thresholds(p_eval, thr, signal_eligible(rows_eval))
        exp_rep = signal_expectancy(pred, rows_eval)
        cls_rep = classifier_report(y_eval, pred)
        return {
            "thresholds": thr,
            "cls": {k: v for k, v in cls_rep.items() if k != "confusion"},
            "confusion": cls_rep["confusion"],
            "exp": exp_rep,
            "score": expectancy_rank_score(exp_rep),
            "pred": pred,
        }


    def fit_fold(
        name: str, fold: str, fit_sel: np.ndarray, calib_sel: np.ndarray, eval_sel: np.ndarray, *, seed: int
    ) -> Dict[str, Any]:
        """Fit one family on ``fit_sel`` rows, return probabilities on calibration and eval rows."""
        mean, std = fit_scaler(all_x[fit_sel])
        pack = make_family(name, seed)
        pack["clf"].fit(scale_x(all_x[fit_sel], mean, std), all_y[fit_sel])
        result = {
            "name": name,
            "fold": fold,
            "clf": pack["clf"],
            "scaler": (mean, std),
            "p_calib": proba_of(pack["clf"], scale_x(all_x[calib_sel], mean, std)),
            "p_eval": proba_of(pack["clf"], scale_x(all_x[eval_sel], mean, std)),
        }
        if fold == "final":
            pack["up"].fit(X_train, R_train[:, 0])
            pack["dn"].fit(X_train, R_train[:, 1])
            result["up"] = pack["up"]
            result["dn"] = pack["dn"]
        return result


    # Fold plan: expanding walk-forward folds inside development, then the final split.
    # Every fold has disjoint fit / calibration / evaluation rows; thresholds are chosen
    # on calibration rows only, so evaluation expectancy is out-of-sample for both the
    # model and the decision rule.
    fold_plan: List[Tuple[str, np.ndarray, np.ndarray, np.ndarray]] = []
    for fold_i, (tr_idx, va_idx) in enumerate(wf_folds, start=1):
        fit_idx, calib_idx = inner_calibration_split(
            tr_idx, calib_fraction=float(CONFIG["calib_fraction"]), purge_bars=int(CONFIG["purge_bars"])
        )
        fold_plan.append((f"wf{fold_i}", rows_in(fit_idx), rows_in(calib_idx), rows_in(va_idx)))
    fold_plan.append(("final", sel_train, sel_val, sel_test))
    for fold, f_sel, c_sel, e_sel in fold_plan:
        print(f"{fold}: fit={int(f_sel.sum())} calib={int(c_sel.sum())} eval={int(e_sel.sum())}")

    FAMILIES: List[str] = list(CONFIG["ensemble"])
    runs: Dict[str, Dict[str, Dict[str, Any]]] = {fold: {} for fold, *_ in fold_plan}
    print("Fitting", FAMILIES, "x", [f for f, *_ in fold_plan], "in parallel...")
    t_start = time.time()
    with ThreadPoolExecutor(max_workers=len(FAMILIES)) as pool:
        futs = {}
        for fold_i, (fold, f_sel, c_sel, e_sel) in enumerate(fold_plan):
            for name in FAMILIES:
                fut = pool.submit(
                    fit_fold, name, fold, f_sel, c_sel, e_sel, seed=int(CONFIG["seed"]) + fold_i
                )
                futs[fut] = (fold, name)
        for fut in as_completed(futs):
            fold, name = futs[fut]
            try:
                runs[fold][name] = fut.result()
                print(f"  done {name:9s} {fold}  ({time.time() - t_start:.0f}s)")
            except Exception as exc:  # keep the bake-off alive if one family fails
                print(f"  FAILED {name} {fold}: {exc!r}")

    # Soft-vote ensemble over the members that succeeded in a fold.
    for fold, f_sel, c_sel, e_sel in fold_plan:
        members = [runs[fold][n] for n in FAMILIES if n in runs[fold]]
        if len(members) >= 2:
            runs[fold][ENSEMBLE_NAME] = {
                "name": ENSEMBLE_NAME,
                "fold": fold,
                "members": [m["name"] for m in members],
                "p_calib": np.mean([m["p_calib"] for m in members], axis=0),
                "p_eval": np.mean([m["p_eval"] for m in members], axis=0),
            }

    CANDIDATES: List[str] = [n for n in FAMILIES + [ENSEMBLE_NAME] if n in runs["final"]]
    scores: Dict[str, Dict[str, Dict[str, Any]]] = {name: {} for name in CANDIDATES}
    for fold, f_sel, c_sel, e_sel in fold_plan:
        rows_c, rows_e, y_e = all_rows[c_sel], all_rows[e_sel], all_y[e_sel]
        for name in CANDIDATES:
            if name not in runs[fold]:
                continue
            r = runs[fold][name]
            scores[name][fold] = score_probas(r["p_calib"], rows_c, r["p_eval"], rows_e, y_e)

    WF_FOLDS: List[str] = [fold for fold, *_ in fold_plan if fold != "final"]


    def walk_forward_summary(name: str) -> Dict[str, float]:
        vals = [scores[name][f]["score"] for f in WF_FOLDS if f in scores[name]]
        finite = [v for v in vals if np.isfinite(v)]
        return {
            "wf_folds_scored": float(len(finite)),
            "wf_mean_score": float(np.mean(finite)) if finite else float("-inf"),
            "wf_positive_share": float(np.mean([v > 0 for v in finite])) if finite else 0.0,
            "wf_min_score": float(np.min(finite)) if finite else float("-inf"),
        }


    leaderboard: Dict[str, Dict[str, Any]] = {}
    for name in CANDIDATES:
        wf = walk_forward_summary(name)
        fin = scores[name].get("final")
        row: Dict[str, Any] = {
            **wf,
            "val_thresholds": fin["thresholds"] if fin else None,
            "wf_folds": {
                f: {"score": scores[name][f]["score"], "exp": scores[name][f]["exp"], "thresholds": scores[name][f]["thresholds"]}
                for f in WF_FOLDS
                if f in scores[name]
            },
        }
        if name != ENSEMBLE_NAME and "up" in runs["final"].get(name, {}):
            r = runs["final"][name]
            up_c = corr(as_1d(r["up"].predict(X_val)), R_val[:, 0])
            dn_c = corr(as_1d(r["dn"].predict(X_val)), R_val[:, 1])
            row["upside_corr"] = up_c
            row["downside_corr"] = dn_c
            row["reg_score"] = float(np.nanmean([up_c, dn_c]))
        leaderboard[name] = row
        print(
            f"{name:9s} WF mean={wf['wf_mean_score']:+.6f} min={wf['wf_min_score']:+.6f} "
            f"pos_share={wf['wf_positive_share']:.2f} folds={wf['wf_folds_scored']:.0f} "
            f"val_thr={row['val_thresholds']} reg={row.get('reg_score', float('nan')):.3f}"
        )


    def _selection_key(name: str) -> Tuple[float, float]:
        wf = leaderboard[name]
        return (wf["wf_mean_score"], wf["wf_min_score"])


    cls_winner_name = max(CANDIDATES, key=_selection_key)
    reg_candidates = [n for n in CANDIDATES if "reg_score" in leaderboard[n]]
    reg_winner_name = max(reg_candidates, key=lambda n: float(leaderboard[n]["reg_score"]))
    cls_winner = {"name": cls_winner_name, **leaderboard[cls_winner_name]}
    reg_winner = {"name": reg_winner_name, "reg_score": leaderboard[reg_winner_name]["reg_score"]}
    print("best classifier (walk-forward):", cls_winner_name, "| best regressor family (val):", reg_winner_name)


Skipping baseline TRAIN


## 14 Held-out evaluation of the winners


In [ ]:
# CELL: EVAL
if not CONFIG.get("run_baseline_train", True):
    print("Skipping baseline EVAL")
else:
    def _unscale_reg(pred: np.ndarray, col: int) -> np.ndarray:
        return as_1d(pred) * float(reg_std[0, col]) + float(reg_mean[0, col])


    rows_test = all_rows[sel_test]
    final_scores = {name: scores[name]["final"] for name in CANDIDATES}
    win = final_scores[cls_winner_name]
    pred_cls = win["pred"]
    cm = np.asarray(win["confusion"])
    print("TEST winner clf=", cls_winner_name, "thresholds=", win["thresholds"], "reg=", reg_winner_name)
    print(
        f"test candles eligible for a signal (TP distance >= {MIN_SIGNAL_TP_PCT:.4%}): "
        f"{float(signal_eligible(rows_test).mean()):.1%}"
    )
    print("test confusion rows=true cols=pred LONG/SHORT/NO_TRADE")
    print(cm)
    print("test signals", {k: v for k, v in win["exp"].items()})

    # Cost sensitivity of the winner's test signals (same decisions, different cost).
    cost_sensitivity = {}
    for c in CONFIG["cost_sensitivity_pct"]:
        rep = signal_expectancy(pred_cls, rows_test, cost=float(c))
        cost_sensitivity[f"{float(c):.4f}"] = {
            "score": expectancy_rank_score(rep),
            "LONG_expectancy": rep["LONG_expectancy"],
            "SHORT_expectancy": rep["SHORT_expectancy"],
        }
        print(f"cost {float(c):.4%}: score={cost_sensitivity[f'{float(c):.4f}']['score']:+.6f}")

    member_test: Dict[str, Dict[str, Any]] = {}
    for name in CANDIDATES:
        s = final_scores[name]
        member_test[name] = {**s["cls"], "exp": s["exp"], "score": s["score"], "thresholds": s["thresholds"]}
        if name != ENSEMBLE_NAME and "up" in runs["final"][name]:
            r = runs["final"][name]
            member_test[name]["upside_corr"] = corr(_unscale_reg(r["up"].predict(X_test), 0), r_test[:, 0])
            member_test[name]["downside_corr"] = corr(_unscale_reg(r["dn"].predict(X_test), 1), r_test[:, 1])
        print(
            name,
            "test",
            {
                "score": round(s["score"], 6) if np.isfinite(s["score"]) else s["score"],
                "LONG": (int(s["exp"]["LONG_n"]), round(s["exp"]["LONG_expectancy"], 6) if np.isfinite(s["exp"]["LONG_expectancy"]) else None),
                "SHORT": (int(s["exp"]["SHORT_n"]), round(s["exp"]["SHORT_expectancy"], 6) if np.isfinite(s["exp"]["SHORT_expectancy"]) else None),
                "P_LONG": round(s["cls"]["precision_LONG"], 3) if np.isfinite(s["cls"]["precision_LONG"]) else None,
                "P_SHORT": round(s["cls"]["precision_SHORT"], 3) if np.isfinite(s["cls"]["precision_SHORT"]) else None,
            },
        )

    best_up = runs["final"][reg_winner_name]["up"]
    best_dn = runs["final"][reg_winner_name]["dn"]
    up_hat = _unscale_reg(best_up.predict(X_test), 0)
    dn_hat = _unscale_reg(best_dn.predict(X_test), 1)
    naive_up_mae = float(np.mean(np.abs(r_test[:, 0] - float(reg_mean[0, 0]))))
    naive_dn_mae = float(np.mean(np.abs(r_test[:, 1] - float(reg_mean[0, 1]))))
    reg_eval = {
        "upside_corr": corr(up_hat, r_test[:, 0]),
        "downside_corr": corr(dn_hat, r_test[:, 1]),
        "upside_mae_skill": 1.0 - float(np.mean(np.abs(r_test[:, 0] - up_hat))) / naive_up_mae,
        "downside_mae_skill": 1.0 - float(np.mean(np.abs(r_test[:, 1] - dn_hat))) / naive_dn_mae,
    }
    print("regression test", reg_eval)

    eval_report = {
        "n_test": int(len(y_test)),
        "best_classifier": cls_winner_name,
        "best_regressor_family": reg_winner_name,
        "decision_rule": {
            "type": "probability_threshold_abstain",
            "thresholds": win["thresholds"],
            "min_signals": int(CONFIG["min_signals"]),
            "min_signal_tp_pct": MIN_SIGNAL_TP_PCT,
            "test_eligible_share": float(signal_eligible(rows_test).mean()),
        },
        "val_leaderboard": leaderboard,
        "test_members": member_test,
        "confusion": cm.tolist(),
        "precision_LONG": win["cls"]["precision_LONG"],
        "precision_SHORT": win["cls"]["precision_SHORT"],
        "n_LONG": win["cls"]["n_LONG"],
        "n_SHORT": win["cls"]["n_SHORT"],
        "balanced_accuracy": win["cls"]["balanced_accuracy"],
        "signal_expectancy": win["exp"],
        "score": win["score"],
        "cost_sensitivity": cost_sensitivity,
        "regression": reg_eval,
        "note": (
            "Classifier chosen by mean walk-forward thresholded expectancy (both sides, "
            f"min {int(CONFIG['min_signals'])} signals per side). Thresholds for the test come from "
            "validation only. Test is scored once. Balanced accuracy is diagnostic only."
        ),
    }
    print(json.dumps({k: v for k, v in eval_report.items() if k not in ("val_leaderboard", "test_members")}, indent=2, default=float))


Skipping baseline EVAL


## 15 Promotion gate


In [ ]:
# CELL: PROMOTE
if not CONFIG.get("run_baseline_train", True):
    print("Skipping baseline PROMOTE")
else:
    def promotion_gate(eval_rep: Dict[str, Any], lb_row: Dict[str, Any]) -> Dict[str, Any]:
        """Decide ``ready_to_promote`` from held-out evidence only.

        Requirements: positive thresholded expectancy on the single held-out test,
        every active side has >= ``min_signals`` trades and t-stat >= ``promote_min_tstat``,
        expectancy stays positive at the highest cost in ``cost_sensitivity_pct``, and the
        walk-forward positive share is >= ``promote_min_wf_positive_share``.
        """
        reasons: List[str] = []
        min_n = int(CONFIG["min_signals"])
        min_t = float(CONFIG["promote_min_tstat"])
        exp = eval_rep["signal_expectancy"]
        active_sides = [s for s in ("LONG", "SHORT") if float(exp.get(f"{s}_n", 0.0)) > 0]
        if not active_sides:
            reasons.append("no_signals_on_test")
        if not (np.isfinite(eval_rep["score"]) and eval_rep["score"] > 0.0):
            reasons.append("test_expectancy_not_positive")
        for side in active_sides:
            if float(exp[f"{side}_n"]) < min_n:
                reasons.append(f"{side.lower()}_too_few_test_signals")
            t = float(exp.get(f"{side}_tstat", float("nan")))
            if not (np.isfinite(t) and t >= min_t):
                reasons.append(f"{side.lower()}_tstat_below_{min_t:g}")
            if not (np.isfinite(exp[f"{side}_expectancy"]) and exp[f"{side}_expectancy"] > 0.0):
                reasons.append(f"{side.lower()}_expectancy_not_positive")
        worst_cost = max(float(c) for c in CONFIG["cost_sensitivity_pct"])
        worst = eval_rep["cost_sensitivity"][f"{worst_cost:.4f}"]["score"]
        if not (np.isfinite(worst) and worst > 0.0):
            reasons.append("fragile_to_higher_cost")
        if float(lb_row.get("wf_positive_share", 0.0)) < float(CONFIG["promote_min_wf_positive_share"]):
            reasons.append("walk_forward_not_consistently_positive")
        if float(lb_row.get("wf_folds_scored", 0.0)) < 2:
            reasons.append("walk_forward_insufficient_folds")
        return {
            "ready_to_promote": len(reasons) == 0,
            "reasons": reasons,
            "test_score": eval_rep["score"],
            "test_signals": {s: exp.get(f"{s}_n") for s in ("LONG", "SHORT")},
            "test_tstats": {s: exp.get(f"{s}_tstat") for s in ("LONG", "SHORT")},
            "wf_positive_share": lb_row.get("wf_positive_share"),
            "wf_mean_score": lb_row.get("wf_mean_score"),
            "worst_cost_score": worst,
        }


    promotion = promotion_gate(eval_report, leaderboard[cls_winner_name])
    ready_to_promote = bool(promotion["ready_to_promote"])
    print(json.dumps(promotion, indent=2, default=float))
    if ready_to_promote:
        print("VERDICT: held-out evidence supports paper-trading this bundle (still research-only here).")
    else:
        print("VERDICT: NOT ready. Do not trade this bundle live. Reasons:", promotion["reasons"])


Skipping baseline PROMOTE


## 16 Export (research only)


In [ ]:
# CELL: EXPORT
if not CONFIG.get("run_baseline_train", True):
    print("Skipping baseline EXPORT")
else:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    final_run = runs["final"][cls_winner_name]
    if cls_winner_name == ENSEMBLE_NAME:
        clf_payload: Any = {
            "members": {m: runs["final"][m]["clf"] for m in final_run["members"]},
            "member_scalers": {m: runs["final"][m]["scaler"] for m in final_run["members"]},
        }
    else:
        clf_payload = {"members": {cls_winner_name: final_run["clf"]}, "member_scalers": {cls_winner_name: final_run["scaler"]}}
    bundle = {
        "best_classifier": cls_winner_name,
        "best_regressor_family": reg_winner_name,
        "clf": clf_payload,
        "thresholds": eval_report["decision_rule"]["thresholds"],
        "up": best_up,
        "dn": best_dn,
        "feat_mean": feat_mean,
        "feat_std": feat_std,
        "reg_mean": reg_mean,
        "reg_std": reg_std,
        "feature_cols": list(FEATURE_COLS),
        "label_names": list(LABEL_NAMES),
    }
    with (EXPORT_DIR / "best_models.pkl").open("wb") as handle:
        pickle.dump(bundle, handle)
    (EXPORT_DIR / "leaderboard.json").write_text(
        json.dumps(leaderboard, indent=2, default=float), encoding="utf-8"
    )
    metadata = {
        "model_family": f"jacksparrow_opportunity_{CONFIG['resolution']}_tabular_ensemble",
        "ensemble": list(CONFIG["ensemble"]),
        "candidates": CANDIDATES,
        "best_classifier": cls_winner_name,
        "best_regressor_family": reg_winner_name,
        "timeframe": str(CONFIG["resolution"]),
        "resolution": str(CONFIG["resolution"]),
        "horizon_bars": int(CONFIG["horizon_bars"]),
        "horizon_minutes": int(CONFIG["horizon_minutes"]),
        "tp_atr_mult": float(CONFIG["tp_atr_mult"]),
        "sl_atr_mult": float(CONFIG["sl_atr_mult"]),
        "round_trip_cost_pct": float(CONFIG["round_trip_cost_pct"]),
        "feature_cols": list(FEATURE_COLS),
        "label_names": list(LABEL_NAMES),
        "decision_rule": {
            **eval_report["decision_rule"],
            "description": (
                f"At a closed {CONFIG['resolution']} candle: if tp_atr_mult * ATR(14) / close < min_signal_tp_pct emit "
                "NO_TRADE (quiet-market guard). Otherwise scale features with the member scaler(s), "
                "average member predict_proba when best_classifier == ENSEMBLE, then emit LONG when "
                "P(LONG) >= thresholds.LONG and P(LONG) > P(SHORT); SHORT symmetrically; else NO_TRADE. "
                "A null threshold disables that side."
            ),
        },
        "ready_to_promote": ready_to_promote,
        "promote_reason": (
            "held_out_gate_passed_paper_trade_first" if ready_to_promote else ";".join(promotion["reasons"])
        ),
        "promotion": promotion,
        "eval": eval_report,
        "label_gate": label_report,
        "walk_forward": {name: leaderboard[name]["wf_folds"] for name in CANDIDATES},
        "config": CONFIG,
    }
    (EXPORT_DIR / "metadata.json").write_text(
        json.dumps(metadata, indent=2, default=float), encoding="utf-8"
    )
    (EXPORT_DIR / "eval_report.json").write_text(
        json.dumps(eval_report, indent=2, default=float), encoding="utf-8"
    )
    print("wrote", EXPORT_DIR / "best_models.pkl")
    print("best classifier", cls_winner_name, "best regressor", reg_winner_name)
    print("ready_to_promote=", ready_to_promote)
    try:
        from google.colab import files
        import shutil

        shutil.make_archive(str(EXPORT_DIR), "zip", root_dir=str(EXPORT_DIR))
        files.download(str(EXPORT_DIR) + ".zip")
    except ImportError:
        print("Not Colab; files remain under", EXPORT_DIR)



Skipping baseline EXPORT


## 17 Feature-label sweep (frozen 30m XGBoost)

XGBoost, 30m bars, and a 24-bar (12 hour) horizon stay fixed. Each remaining
label spec yields an opportunity head and a direction head; the combined
policy is ranked on expanding walk-forward R-multiples. Stage A keeps specs
with both sides above the minority floor (L5 is direction-only). Stage B
screens F0–F7 plus leave-one-family-out. Column drop-one and permutation run
on the top two specs. Baseline TRAIN/EVAL/EXPORT is skipped.


In [ ]:
# CELL: FL_SPECS
"""Feature-label sweep for the frozen 30m / 24-bar XGBoost cell.

Fixes look-ahead in L5 eligibility and L7 scoring, ranks on R-multiples, splits
each spec into opportunity vs direction heads, and screens families plus
individual columns. Injected into the sweep notebook as ``# CELL: FL_SPECS``.
"""

from __future__ import annotations

import json
import sys
from pathlib import Path
from typing import Any, Callable, Dict, List, Mapping, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

try:
    sys.stdout.reconfigure(line_buffering=True)
except Exception:
    pass

LABEL_LONG = 0
LABEL_SHORT = 1
LABEL_NO_TRADE = 2
LABEL_UNLABELED = -1
TP_FIRST = 0
SL_FIRST = 1
NEITHER = 2
AMBIGUOUS = 3
FROZEN_HORIZON_BARS = 24
FROZEN_RESOLUTION = "30m"
STAGE_B_TREES = 400
STAGE_C_TREES = 400
STAGE_C_TOP_K = 3
SCREEN_SEED = 42
FINALIST_SEEDS: Tuple[int, ...] = (42, 43, 44)
OPP_DEGENERATE_LOW = 0.05
OPP_DEGENERATE_HIGH = 0.95
R_EPS = 1e-12

CURRENT_FEATURE_COLS: Tuple[str, ...] = (
    "ret_1",
    "ret_3_atr",
    "ret_6_atr",
    "ret_12_atr",
    "ret_24_atr",
    "ret_48_atr",
    "ret_144_atr",
    "atr_pct",
    "atr_ratio_84",
    "rv_12_48",
    "rv_48_144",
    "rsi_7",
    "rsi_14",
    "rsi_28",
    "ema_dist_20_atr",
    "ema_slope_20_atr",
    "ema_dist_50_atr",
    "ema_slope_50_atr",
    "ema_dist_200_atr",
    "ema_slope_200_atr",
    "candle_body",
    "upper_wick",
    "lower_wick",
    "range_atr",
    "volume_z_20",
    "volume_z_100",
    "range_pos_24",
    "dist_high_24_atr",
    "dist_low_24_atr",
    "range_pos_72",
    "dist_high_72_atr",
    "dist_low_72_atr",
    "range_pos_144",
    "dist_high_144_atr",
    "dist_low_144_atr",
    "streak",
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "ret_lag1_atr",
    "ret_lag2_atr",
    "ret_lag3_atr",
    "ret_lag4_atr",
    "ret_lag5_atr",
    "ret_lag6_atr",
)

F7_COLS: Tuple[str, ...] = (
    "efficiency_24",
    "rv24_over_tp",
    "rv_24",
    "bars_since_2atr_swing",
)

FEATURE_FAMILIES: Dict[str, Tuple[str, ...]] = {
    "F1": (
        "ret_1",
        "ret_3_atr",
        "ret_6_atr",
        "ret_12_atr",
        "ret_24_atr",
        "ret_48_atr",
        "ret_144_atr",
        "streak",
        "ret_lag1_atr",
        "ret_lag2_atr",
        "ret_lag3_atr",
        "ret_lag4_atr",
        "ret_lag5_atr",
        "ret_lag6_atr",
    ),
    "F2": (
        "rsi_7",
        "rsi_14",
        "rsi_28",
        "ema_dist_20_atr",
        "ema_slope_20_atr",
        "ema_dist_50_atr",
        "ema_slope_50_atr",
        "ema_dist_200_atr",
        "ema_slope_200_atr",
    ),
    "F3": ("atr_pct", "atr_ratio_84", "rv_12_48", "rv_48_144"),
    "F4": (
        "range_pos_24",
        "dist_high_24_atr",
        "dist_low_24_atr",
        "range_pos_72",
        "dist_high_72_atr",
        "dist_low_72_atr",
        "range_pos_144",
        "dist_high_144_atr",
        "dist_low_144_atr",
    ),
    "F5": (
        "candle_body",
        "upper_wick",
        "lower_wick",
        "range_atr",
        "volume_z_20",
        "volume_z_100",
    ),
    "F6": ("hour_sin", "hour_cos", "dow_sin", "dow_cos"),
    "F7": F7_COLS,
}

FEATURE_FAMILY_ORDER: Tuple[str, ...] = (
    "F0",
    "F1",
    "F2",
    "F3",
    "F4",
    "F5",
    "F6",
    "F7",
)

LABEL_SPECS: Tuple[Dict[str, Any], ...] = (
    {
        "id": "L0",
        "kind": "barrier",
        "group": "L0",
        "tp_atr_mult": 4.0,
        "sl_atr_mult": 2.0,
        "horizon_bars": FROZEN_HORIZON_BARS,
    },
    {
        "id": "L1_2_2",
        "kind": "barrier",
        "group": "L1",
        "tp_atr_mult": 2.0,
        "sl_atr_mult": 2.0,
        "horizon_bars": FROZEN_HORIZON_BARS,
    },
    {
        "id": "L1_3_3",
        "kind": "barrier",
        "group": "L1",
        "tp_atr_mult": 3.0,
        "sl_atr_mult": 3.0,
        "horizon_bars": FROZEN_HORIZON_BARS,
    },
    {
        "id": "L1_1p5_1p5",
        "kind": "barrier",
        "group": "L1",
        "tp_atr_mult": 1.5,
        "sl_atr_mult": 1.5,
        "horizon_bars": FROZEN_HORIZON_BARS,
    },
    {
        "id": "L2_2_1",
        "kind": "barrier",
        "group": "L2",
        "tp_atr_mult": 2.0,
        "sl_atr_mult": 1.0,
        "horizon_bars": FROZEN_HORIZON_BARS,
    },
    {
        "id": "L2_3_1p5",
        "kind": "barrier",
        "group": "L2",
        "tp_atr_mult": 3.0,
        "sl_atr_mult": 1.5,
        "horizon_bars": FROZEN_HORIZON_BARS,
    },
    {
        "id": "L3",
        "kind": "barrier",
        "group": "L3",
        "tp_atr_mult": 6.0,
        "sl_atr_mult": 3.0,
        "horizon_bars": FROZEN_HORIZON_BARS,
    },
    {
        "id": "L4",
        "kind": "percent",
        "group": "L4",
        "tp_pct": 0.008,
        "sl_pct": 0.004,
        "horizon_bars": FROZEN_HORIZON_BARS,
    },
    {
        "id": "L5",
        "kind": "sign",
        "group": "L5",
        "horizon_bars": FROZEN_HORIZON_BARS,
    },
    {
        "id": "L6",
        "kind": "regression",
        "group": "L6",
        "tp_atr_mult": 4.0,
        "sl_atr_mult": 2.0,
        "horizon_bars": FROZEN_HORIZON_BARS,
    },
    {
        "id": "L7",
        "kind": "decisive",
        "group": "L7",
        "tp_atr_mult": 4.0,
        "sl_atr_mult": 2.0,
        "horizon_bars": FROZEN_HORIZON_BARS,
    },
    {
        "id": "L8",
        "kind": "meta",
        "group": "L8",
        "tp_atr_mult": 4.0,
        "sl_atr_mult": 2.0,
        "horizon_bars": FROZEN_HORIZON_BARS,
    },
)

STAGE_B_REQUIRED: Tuple[str, ...] = (
    "label_id",
    "feature_family",
    "wf_r_expectancy",
    "wf_tstat",
    "wf_positive_share",
    "n_LONG",
    "n_SHORT",
)
STAGE_B_FORBIDDEN: Tuple[str, ...] = (
    "test_score",
    "test_precision_LONG",
    "test_precision_SHORT",
    "test_expectancy",
    "test_r_expectancy",
    "n_test",
    "test_n_LONG",
    "test_n_SHORT",
)

DIAGNOSIS_BARRIER_ABSENT = (
    "the 30m series over 24 bars does not contain this barrier"
)
DIAGNOSIS_FIRST_TOUCH = "the 4 ATR / 2 ATR first-touch rule is the failure"
DIAGNOSIS_VOL_FEATURES = (
    "the missing information is vol relative to the barriers"
)
DIAGNOSIS_SHORT_ABSENT = (
    "the short side is still absent; report sides separately"
)
DIAGNOSIS_INCONCLUSIVE = "inconclusive: no diagnosis rule matched"
DIAGNOSIS_NULL = "best pair does not beat the shuffled-label null"


def label_spec_by_id(spec_id: str) -> Dict[str, Any]:
    """Return a copy of one label spec."""
    for spec in LABEL_SPECS:
        if spec["id"] == spec_id:
            return dict(spec)
    raise KeyError(f"Unknown label spec {spec_id!r}")


def barrier_label_specs() -> Tuple[Dict[str, Any], ...]:
    """Specs that use a first-touch ATR or percent barrier on 24 forward bars."""
    kinds = {"barrier", "percent", "regression", "decisive", "meta"}
    return tuple(dict(s) for s in LABEL_SPECS if s["kind"] in kinds)


def family_columns(family_id: str) -> Tuple[str, ...]:
    """Column names for one feature family. F0 is the current set plus F7."""
    if family_id == "F0":
        return CURRENT_FEATURE_COLS + F7_COLS
    if family_id.startswith("F0_minus_"):
        dropped = family_id.split("F0_minus_", 1)[1]
        drop_set = set(family_columns(dropped))
        return tuple(c for c in family_columns("F0") if c not in drop_set)
    if family_id not in FEATURE_FAMILIES:
        raise KeyError(f"Unknown feature family {family_id!r}")
    return FEATURE_FAMILIES[family_id]


def disjoint_family_ids() -> Tuple[str, ...]:
    """Family ids that must be pairwise disjoint (every family except F0)."""
    return tuple(fid for fid in FEATURE_FAMILY_ORDER if fid != "F0")


def screen_column_sets() -> List[Tuple[str, Tuple[str, ...]]]:
    """Families F0–F7 plus leave-one-family-out from F0."""
    sets: List[Tuple[str, Tuple[str, ...]]] = []
    for fam in FEATURE_FAMILY_ORDER:
        sets.append((fam, family_columns(fam)))
    for fam in disjoint_family_ids():
        name = f"F0_minus_{fam}"
        sets.append((name, family_columns(name)))
    return sets


def spec_f7_kwargs(spec: Mapping[str, Any]) -> Dict[str, Any]:
    """Per-spec distances so F7 is not hard-wired to 4/2."""
    horizon = int(spec.get("horizon_bars", FROZEN_HORIZON_BARS))
    if str(spec.get("kind")) == "percent":
        return {
            "tp_pct": float(spec["tp_pct"]),
            "sl_pct": float(spec["sl_pct"]),
            "horizon_bars": horizon,
        }
    if str(spec.get("kind")) == "sign":
        return {"tp_atr_mult": 1.0, "sl_atr_mult": 1.0, "horizon_bars": horizon}
    return {
        "tp_atr_mult": float(spec.get("tp_atr_mult", 4.0)),
        "sl_atr_mult": float(spec.get("sl_atr_mult", 2.0)),
        "horizon_bars": horizon,
    }


def add_f7_features(
    df: pd.DataFrame,
    *,
    tp_atr_mult: float = 4.0,
    sl_atr_mult: float = 2.0,
    tp_pct: Optional[float] = None,
    sl_pct: Optional[float] = None,
    horizon_bars: int = FROZEN_HORIZON_BARS,
) -> pd.DataFrame:
    """Append causal vol-versus-barrier columns. Uses only bars at or before t."""
    out = df.copy()
    close = out["close"].astype(np.float64)
    atr = (
        out["atr"].astype(np.float64)
        if "atr" in out.columns
        else pd.Series(np.nan, index=out.index)
    )
    k = int(horizon_bars)
    log_ret = np.log(close.replace(0.0, np.nan)).diff()
    rv_h = log_ret.rolling(k, min_periods=k).std()
    net = close - close.shift(k)
    path = close.diff().abs().rolling(k, min_periods=k).sum()
    atr_safe = atr.replace(0.0, np.nan)
    close_safe = close.replace(0.0, np.nan)
    atr_pct = atr_safe / close_safe
    if tp_pct is not None:
        tp_dist = pd.Series(float(tp_pct), index=out.index)
    else:
        tp_dist = float(tp_atr_mult) * atr_pct
    move_atr = close.diff().abs() / atr_safe
    is_swing = (move_atr >= 2.0).to_numpy()
    idx = np.arange(len(out), dtype=np.float64)
    last = np.where(is_swing, idx, np.nan)
    last_swing = pd.Series(last, index=out.index).ffill()
    out["efficiency_24"] = net / path.replace(0.0, np.nan)
    out["rv24_over_tp"] = rv_h / tp_dist.replace(0.0, np.nan)
    out["rv_24"] = rv_h
    out["bars_since_2atr_swing"] = idx - last_swing.to_numpy(dtype=np.float64)
    if sl_pct is None:
        _ = sl_atr_mult
    return out


def r_multiple(pnl: np.ndarray, sl_pct: np.ndarray) -> np.ndarray:
    """P&L in stop-distance units. Causal sl_pct only."""
    sl = np.asarray(sl_pct, dtype=np.float64)
    pnl_arr = np.asarray(pnl, dtype=np.float64)
    return pnl_arr / np.where(np.abs(sl) > R_EPS, sl, np.nan)


def _named_labels(labels: np.ndarray) -> np.ndarray:
    names = np.array(["UNLABELED"] * labels.shape[0], dtype=object)
    names[labels == LABEL_LONG] = "LONG"
    names[labels == LABEL_SHORT] = "SHORT"
    names[labels == LABEL_NO_TRADE] = "NO_TRADE"
    return names


def _causal_atr_pct(df: pd.DataFrame) -> np.ndarray:
    close = df["close"].to_numpy(dtype=np.float64)
    if "atr" in df.columns:
        atr = df["atr"].to_numpy(dtype=np.float64)
    else:
        atr = np.full(len(df), np.nan)
    with np.errstate(invalid="ignore", divide="ignore"):
        return atr / np.where(np.abs(close) > R_EPS, close, np.nan)


def _apply_sign_labels(df: pd.DataFrame, *, horizon_bars: int) -> pd.DataFrame:
    """LONG/SHORT from the sign of the 24-bar close-to-close return.

    ``tp_pct`` / ``sl_pct`` are causal ATR fractions, not the forward move.
    """
    out = df.copy().reset_index(drop=True)
    n = len(out)
    k = int(horizon_bars)
    close = out["close"].to_numpy(dtype=np.float64)
    fwd = np.full(n, np.nan, dtype=np.float64)
    if n > k:
        fwd[: n - k] = close[k:]
    with np.errstate(invalid="ignore", divide="ignore"):
        timeout = (fwd - close) / np.where(np.abs(close) > 1e-12, close, np.nan)
    valid = np.isfinite(timeout)
    labels = np.full(n, LABEL_UNLABELED, dtype=np.int64)
    labels[valid & (timeout > 0.0)] = LABEL_LONG
    labels[valid & (timeout < 0.0)] = LABEL_SHORT
    labels[valid & (timeout == 0.0)] = LABEL_NO_TRADE
    long_code = np.full(n, np.nan, dtype=np.float64)
    short_code = np.full(n, np.nan, dtype=np.float64)
    long_code[valid & (timeout > 0.0)] = float(TP_FIRST)
    long_code[valid & (timeout < 0.0)] = float(SL_FIRST)
    long_code[valid & (timeout == 0.0)] = float(NEITHER)
    short_code[valid & (timeout < 0.0)] = float(TP_FIRST)
    short_code[valid & (timeout > 0.0)] = float(SL_FIRST)
    short_code[valid & (timeout == 0.0)] = float(NEITHER)
    atr_pct = _causal_atr_pct(out)
    out["trade_label"] = labels
    out["trade_name"] = _named_labels(labels)
    out["long_pnl_pct"] = np.where(valid, timeout, np.nan)
    out["short_pnl_pct"] = np.where(valid, -timeout, np.nan)
    out["long_code"] = long_code
    out["short_code"] = short_code
    out["long_result"] = np.where(valid, (labels == LABEL_LONG).astype(np.float64), np.nan)
    out["short_result"] = np.where(valid, (labels == LABEL_SHORT).astype(np.float64), np.nan)
    out["timeout_ret"] = np.where(valid, timeout, np.nan)
    out["tp_pct"] = np.where(valid, atr_pct, np.nan)
    out["sl_pct"] = np.where(valid, atr_pct, np.nan)
    if "upside" not in out.columns:
        out["upside"] = np.where(valid, np.maximum(timeout, 0.0), np.nan)
    if "downside" not in out.columns:
        out["downside"] = np.where(valid, np.maximum(-timeout, 0.0), np.nan)
    return out


def _apply_meta_labels(
    labeled: pd.DataFrame, *, cost: float, momentum_col: str = "ret_1"
) -> pd.DataFrame:
    """Keep LONG/SHORT only when a simple momentum entry would have paid the cost."""
    out = labeled.copy()
    if momentum_col in out.columns:
        mom = out[momentum_col].to_numpy(dtype=np.float64)
    else:
        close = out["close"].to_numpy(dtype=np.float64)
        prev = np.where(close[:-1] == 0, np.nan, close[:-1])
        mom = np.concatenate([[np.nan], np.diff(close) / prev])
    long_pnl = out["long_pnl_pct"].to_numpy(dtype=np.float64)
    short_pnl = out["short_pnl_pct"].to_numpy(dtype=np.float64)
    old = out["trade_label"].to_numpy(dtype=np.int64)
    labels = np.full(len(out), LABEL_UNLABELED, dtype=np.int64)
    usable = old >= 0
    labels[usable] = LABEL_NO_TRADE
    long_win = (
        usable
        & np.isfinite(mom)
        & (mom > 0.0)
        & np.isfinite(long_pnl)
        & (long_pnl - float(cost) > 0.0)
    )
    short_win = (
        usable
        & np.isfinite(mom)
        & (mom < 0.0)
        & np.isfinite(short_pnl)
        & (short_pnl - float(cost) > 0.0)
    )
    labels[long_win] = LABEL_LONG
    labels[short_win] = LABEL_SHORT
    out["trade_label"] = labels
    out["trade_name"] = _named_labels(labels)
    return out


def attach_heads(
    labeled: pd.DataFrame, spec: Mapping[str, Any]
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """Add opportunity / direction columns. Timeouts stay scored as NO_TRADE."""
    out = labeled.copy()
    labels = out["trade_label"].to_numpy(dtype=np.int64)
    long_c = out["long_code"].to_numpy(dtype=np.float64)
    short_c = out["short_code"].to_numpy(dtype=np.float64)
    valid = labels >= 0
    opp = np.full(len(out), np.nan, dtype=np.float64)
    tp_any = (long_c == float(TP_FIRST)) | (short_c == float(TP_FIRST))
    opp[valid] = tp_any[valid].astype(np.float64)
    out["opp_label"] = np.where(valid, opp, -1.0)
    dir_train = valid & ((labels == LABEL_LONG) | (labels == LABEL_SHORT))
    if str(spec.get("kind")) == "decisive":
        timeout = (
            np.isfinite(long_c)
            & np.isfinite(short_c)
            & (long_c == float(NEITHER))
            & (short_c == float(NEITHER))
        )
        labels = labels.copy()
        labels[timeout & (labels == LABEL_UNLABELED)] = LABEL_NO_TRADE
        out["trade_label"] = labels
        out["trade_name"] = _named_labels(labels)
        valid = labels >= 0
        dir_train = valid & ((labels == LABEL_LONG) | (labels == LABEL_SHORT))
        opp = np.full(len(out), np.nan, dtype=np.float64)
        opp[valid] = tp_any[valid].astype(np.float64)
        out["opp_label"] = np.where(valid, opp, -1.0)
    out["dir_train_mask"] = dir_train
    rate = float(np.nanmean(opp[valid])) if valid.any() else float("nan")
    degen = bool(
        np.isfinite(rate) and (rate >= OPP_DEGENERATE_HIGH or rate <= OPP_DEGENERATE_LOW)
    )
    out["opportunity_degenerate"] = degen
    return out, {"opportunity_rate": rate, "opportunity_degenerate": degen}


def apply_label_spec(
    featured: pd.DataFrame,
    spec: Mapping[str, Any],
    compute_opportunity_labels: Callable[..., pd.DataFrame],
    *,
    atr_period: int = 14,
    round_trip_cost_pct: float = 0.0016,
) -> pd.DataFrame:
    """Rebuild y on the same 24-bar forward path according to ``spec``."""
    horizon = int(spec["horizon_bars"])
    kind = str(spec["kind"])
    if kind == "sign":
        labeled = _apply_sign_labels(featured, horizon_bars=horizon)
    elif kind == "percent":
        tmp = featured.copy()
        tmp["atr"] = tmp["close"].astype(np.float64)
        labeled = compute_opportunity_labels(
            tmp,
            tp_mult=float(spec["tp_pct"]),
            sl_mult=float(spec["sl_pct"]),
            horizon_bars=horizon,
            atr_period=int(atr_period),
        )
        if "atr" in featured.columns:
            labeled["atr"] = featured["atr"].to_numpy(dtype=np.float64)
    else:
        labeled = compute_opportunity_labels(
            featured,
            tp_mult=float(spec["tp_atr_mult"]),
            sl_mult=float(spec["sl_atr_mult"]),
            horizon_bars=horizon,
            atr_period=int(atr_period),
        )
        if kind == "meta":
            labeled = _apply_meta_labels(labeled, cost=float(round_trip_cost_pct))
    labeled, _heads = attach_heads(labeled, spec)
    return labeled


def l7_timeouts_are_scored(labeled: pd.DataFrame) -> bool:
    """True when dual-timeout bars remain in the eval row set as NO_TRADE."""
    long_c = labeled["long_code"].to_numpy(dtype=np.float64)
    short_c = labeled["short_code"].to_numpy(dtype=np.float64)
    labels = labeled["trade_label"].to_numpy(dtype=np.int64)
    timeout = (
        np.isfinite(long_c)
        & np.isfinite(short_c)
        & (long_c == float(NEITHER))
        & (short_c == float(NEITHER))
    )
    if not timeout.any():
        return True
    return bool(np.all(labels[timeout] == LABEL_NO_TRADE))


def stage_a_should_keep(
    report: Mapping[str, Any],
    *,
    minority_rate: float = 0.15,
    allow_degenerate_opp: bool = False,
) -> bool:
    """Keep if both sides exist above the minority floor.

    Exclusive-winner expectancy is reported, never used as the keep rule.
    Degenerate opportunity (~0 or ~1) drops the spec unless it is a
    direction-only label (L5 / ``allow_degenerate_opp``).
    """
    mix = report.get("class_mix") or {}
    long_r = float(mix.get("LONG", report.get("base_long_tp_rate", 0.0)) or 0.0)
    short_r = float(mix.get("SHORT", report.get("base_short_tp_rate", 0.0)) or 0.0)
    if long_r < float(minority_rate) or short_r < float(minority_rate):
        return False
    if allow_degenerate_opp:
        return True
    degen = report.get("opportunity_degenerate")
    if degen is True:
        return False
    rate = report.get("opportunity_rate")
    if rate is not None and np.isfinite(float(rate)):
        value = float(rate)
        if value >= OPP_DEGENERATE_HIGH or value <= OPP_DEGENERATE_LOW:
            return False
    return True


def validate_stage_b_row(row: Mapping[str, Any]) -> bool:
    """Accept a screen row with walk-forward R fields and no test metrics."""
    keys = set(row.keys())
    missing = [k for k in STAGE_B_REQUIRED if k not in keys]
    if missing:
        raise ValueError(f"Stage B row missing fields: {missing}")
    forbidden = [k for k in STAGE_B_FORBIDDEN if k in keys]
    if forbidden:
        raise ValueError(f"Stage B row includes test metrics: {sorted(forbidden)}")
    return True


def select_top_pairs(
    rows: Sequence[Mapping[str, Any]], *, k: int = STAGE_C_TOP_K
) -> List[Dict[str, Any]]:
    """Highest walk-forward R-expectancy first. Percent P&L is not used."""

    def _key(row: Mapping[str, Any]) -> float:
        raw = row.get("wf_r_expectancy", row.get("val_expectancy", float("-inf")))
        try:
            val = float(raw)
        except (TypeError, ValueError):
            return float("-inf")
        return val if np.isfinite(val) else float("-inf")

    ranked = sorted((dict(r) for r in rows), key=_key, reverse=True)
    return ranked[: max(int(k), 0)]


def _rows_for(
    rows: Sequence[Mapping[str, Any]],
    *,
    label_id: Optional[str] = None,
    group: Optional[str] = None,
) -> List[Mapping[str, Any]]:
    out: List[Mapping[str, Any]] = []
    for row in rows:
        if label_id is not None and str(row.get("label_id")) != label_id:
            continue
        if group is not None and str(row.get("label_group", "")) != group:
            lid = str(row.get("label_id", ""))
            if not (lid == group or lid.startswith(f"{group}_")):
                continue
        out.append(row)
    return out


def _wf_positive(row: Mapping[str, Any]) -> bool:
    r = float(row.get("wf_r_expectancy", float("nan")))
    return bool(np.isfinite(r) and r > 0.0)


def diagnose_sweep(
    rows: Sequence[Mapping[str, Any]],
    *,
    shuffle_best_r: Optional[float] = None,
) -> str:
    """Name the failure from walk-forward R-expectancy, not leaked val scores."""
    if not rows:
        return DIAGNOSIS_INCONCLUSIVE
    finite = [
        r
        for r in rows
        if np.isfinite(float(r.get("wf_r_expectancy", float("nan"))))
    ]
    if not finite:
        return DIAGNOSIS_INCONCLUSIVE
    best_r = max(float(r["wf_r_expectancy"]) for r in finite)
    if shuffle_best_r is not None and np.isfinite(float(shuffle_best_r)):
        if best_r <= float(shuffle_best_r):
            return DIAGNOSIS_NULL
    l0 = _rows_for(finite, label_id="L0")
    l0_pos = any(_wf_positive(r) for r in l0)
    l0_by_fam = {str(r.get("feature_family")): r for r in l0}
    vol_pos = any(
        _wf_positive(l0_by_fam[f]) for f in ("F3", "F7") if f in l0_by_fam
    )
    others_pos = any(
        _wf_positive(r)
        for fid, r in l0_by_fam.items()
        if fid not in {"F3", "F7"}
    )
    if l0 and vol_pos and not others_pos:
        return DIAGNOSIS_VOL_FEATURES
    alt = [
        r
        for r in finite
        if str(r.get("label_id")) == "L5" or str(r.get("label_id", "")).startswith("L1")
    ]
    if (not l0_pos) and any(_wf_positive(r) for r in alt):
        return DIAGNOSIS_FIRST_TOUCH
    if l0 and not l0_pos and best_r <= 0.0:
        return DIAGNOSIS_BARRIER_ABSENT
    dir_aucs = [
        float(r["dir_auc"])
        for r in finite
        if r.get("dir_auc") is not None
        and np.isfinite(float(r.get("dir_auc", float("nan"))))
    ]
    if dir_aucs and float(np.mean(dir_aucs)) < 0.52:
        return DIAGNOSIS_SHORT_ABSENT
    return DIAGNOSIS_INCONCLUSIVE


def block_shuffle(values: np.ndarray, *, block: int, rng: np.random.Generator) -> np.ndarray:
    """Permute labels in horizon-length blocks so serial dependence is kept."""
    arr = np.asarray(values)
    n = arr.shape[0]
    k = max(int(block), 1)
    parts = [arr[i : i + k] for i in range(0, n, k)]
    order = rng.permutation(len(parts))
    return np.concatenate([parts[i] for i in order], axis=0)[:n]


def _fit_scaler(x: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    mean = x.mean(axis=0, keepdims=True)
    std = x.std(axis=0, keepdims=True)
    std = np.where(std < 1e-6, 1.0, std)
    return mean.astype(np.float32), std.astype(np.float32)


def _scale_x(x: np.ndarray, mean: np.ndarray, std: np.ndarray) -> np.ndarray:
    return ((x - mean) / std).astype(np.float32)


def _xgb_kwargs(config: Mapping[str, Any], n_estimators: int, seed: int) -> Dict[str, Any]:
    return dict(
        n_estimators=int(n_estimators),
        max_depth=int(config["max_depth"]),
        learning_rate=float(config["learning_rate"]),
        min_child_weight=int(config["min_leaf"]),
        subsample=0.8,
        colsample_bytree=0.7,
        reg_lambda=5.0,
        tree_method="hist",
        n_jobs=int(config.get("threads_per_model", 2)),
        verbosity=0,
        random_state=int(seed),
    )


def _align_binary_proba(clf: Any, x: np.ndarray) -> np.ndarray:
    raw = np.asarray(clf.predict_proba(x), dtype=np.float64)
    if raw.ndim == 1:
        return raw.reshape(-1)
    classes = list(np.asarray(getattr(clf, "classes_", [0, 1])).reshape(-1))
    if 1 in classes:
        return raw[:, classes.index(1)]
    if True in classes:
        return raw[:, classes.index(True)]
    return raw[:, -1]


def _align_dir_proba(clf: Any, x: np.ndarray) -> np.ndarray:
    n = x.shape[0]
    full = np.zeros((n, 3), dtype=np.float64)
    if clf is None:
        full[:] = 1.0 / 3.0
        return full
    p_long = _align_binary_proba(clf, x)
    full[:, LABEL_LONG] = p_long
    full[:, LABEL_SHORT] = 1.0 - p_long
    return full


def _roc_auc(y: np.ndarray, p: np.ndarray) -> float:
    y = np.asarray(y).reshape(-1)
    p = np.asarray(p, dtype=np.float64).reshape(-1)
    mask = np.isfinite(p) & np.isfinite(y)
    y, p = y[mask], p[mask]
    if y.size < 8 or len(np.unique(y)) < 2:
        return float("nan")
    try:
        from sklearn.metrics import roc_auc_score

        return float(roc_auc_score(y, p))
    except Exception:
        return float("nan")


def _eligible_mask(
    frame: pd.DataFrame, index: np.ndarray, config: Mapping[str, Any]
) -> np.ndarray:
    min_tp = float(config["min_tp_cost_ratio"]) * float(config["round_trip_cost_pct"])
    if "tp_pct" not in frame.columns:
        return np.ones(index.shape[0], dtype=bool)
    tp = frame["tp_pct"].to_numpy(dtype=np.float64)[index]
    finite = np.isfinite(tp)
    if not finite.any():
        return np.ones(index.shape[0], dtype=bool)
    return finite & (tp >= min_tp)


def _usable_rows(frame: pd.DataFrame, feature_cols: Sequence[str]) -> np.ndarray:
    feats = frame.loc[:, list(feature_cols)].to_numpy(dtype=np.float64)
    y = frame["trade_label"].to_numpy(dtype=np.int64)
    ok = (y >= 0) & np.isfinite(feats).all(axis=1)
    return np.flatnonzero(ok).astype(np.int64)


def _selector(all_rows: np.ndarray, split_idx: np.ndarray, n_frame: int) -> np.ndarray:
    allowed = np.zeros(n_frame, dtype=bool)
    idx = np.asarray(split_idx, dtype=np.int64)
    idx = idx[(idx >= 0) & (idx < n_frame)]
    allowed[idx] = True
    return allowed[all_rows]


def weighted_r_score(
    pred: np.ndarray,
    pnl_long: np.ndarray,
    pnl_short: np.ndarray,
    sl: np.ndarray,
    min_signals: int,
) -> Dict[str, float]:
    """Trade-count-weighted R-expectancy. No n < min_signals fallback."""
    parts: List[np.ndarray] = []
    n_long = 0
    n_short = 0
    for cls, pnl in ((LABEL_LONG, pnl_long), (LABEL_SHORT, pnl_short)):
        take = pred == cls
        r = r_multiple(pnl[take], sl[take])
        r = r[np.isfinite(r)]
        if cls == LABEL_LONG:
            n_long = int(r.size)
        else:
            n_short = int(r.size)
        if r.size >= int(min_signals):
            parts.append(r)
    if not parts:
        return {
            "r_expectancy": float("-inf"),
            "tstat": float("nan"),
            "n_LONG": float(n_long),
            "n_SHORT": float(n_short),
        }
    pooled = np.concatenate(parts)
    mean_r = float(pooled.mean())
    n = int(pooled.size)
    std = float(pooled.std(ddof=1)) if n > 1 else float("nan")
    tstat = (
        float(mean_r / (std / np.sqrt(n))) if n > 1 and std > 0 else float("nan")
    )
    return {
        "r_expectancy": mean_r,
        "tstat": tstat,
        "n_LONG": float(n_long),
        "n_SHORT": float(n_short),
    }


def _class_precision(y_true: np.ndarray, y_pred: np.ndarray, cls: int) -> float:
    pred = y_pred == cls
    n = int(pred.sum())
    if n == 0:
        return float("nan")
    return float(np.mean(y_true[pred] == cls))


def _fit_opp_dir(
    x_fit: np.ndarray,
    opp_fit: np.ndarray,
    dir_y: np.ndarray,
    dir_ok: np.ndarray,
    *,
    config: Mapping[str, Any],
    n_estimators: int,
    seed: int,
    skip_opp: bool,
) -> Tuple[Any, Any]:
    from xgboost import XGBClassifier

    kwargs = _xgb_kwargs(config, n_estimators, seed)
    opp_clf: Any = None
    if not skip_opp and len(np.unique(opp_fit[np.isfinite(opp_fit)])) >= 2:
        opp_clf = XGBClassifier(
            objective="binary:logistic", eval_metric="logloss", **kwargs
        )
        y_opp = (opp_fit >= 0.5).astype(np.int64)
        opp_clf.fit(x_fit, y_opp)
    dir_clf: Any = None
    mask = np.asarray(dir_ok, dtype=bool).reshape(-1)
    if mask.any() and len(np.unique(dir_y[mask])) >= 2:
        y_dir = (np.asarray(dir_y)[mask] == LABEL_LONG).astype(np.int64)
        dir_clf = XGBClassifier(
            objective="binary:logistic", eval_metric="logloss", **kwargs
        )
        dir_clf.fit(x_fit[mask], y_dir)
    return opp_clf, dir_clf


def _predict_combined(
    opp_clf: Any,
    dir_clf: Any,
    x: np.ndarray,
    skip_opp: bool,
) -> Tuple[np.ndarray, np.ndarray]:
    n = x.shape[0]
    if skip_opp or opp_clf is None:
        p_opp = np.ones(n, dtype=np.float64)
    else:
        p_opp = _align_binary_proba(opp_clf, x)
    p_dir = _align_dir_proba(dir_clf, x)
    return p_opp, p_dir


def _choose_combined_thresholds(
    p_opp: np.ndarray,
    p_dir: np.ndarray,
    pnl_long: np.ndarray,
    pnl_short: np.ndarray,
    sl: np.ndarray,
    eligible: np.ndarray,
    *,
    quantiles: Sequence[float],
    min_signals: int,
    skip_opp: bool,
) -> Dict[str, Optional[float]]:
    best: Dict[str, Optional[float]] = {"OPP": 0.0, "LONG": None, "SHORT": None}
    best_e = float("-inf")
    opp_qs: Sequence[float] = (0.0,) if skip_opp else (0.0, 0.5, 0.8, 0.9)
    p_long, p_short = p_dir[:, LABEL_LONG], p_dir[:, LABEL_SHORT]
    for q_opp in opp_qs:
        if skip_opp:
            thr_opp = 0.0
        elif eligible.any():
            thr_opp = float(np.quantile(p_opp[eligible], q_opp))
        else:
            thr_opp = 0.0
        opp_ok = eligible & (p_opp >= thr_opp)
        for q in quantiles:
            thr_l = float(np.quantile(p_long[opp_ok], float(q))) if opp_ok.any() else 1.0
            thr_s = float(np.quantile(p_short[opp_ok], float(q))) if opp_ok.any() else 1.0
            pred = np.full(p_opp.shape[0], LABEL_NO_TRADE, dtype=np.int64)
            pred[opp_ok & (p_long >= thr_l) & (p_long > p_short)] = LABEL_LONG
            pred[opp_ok & (p_short >= thr_s) & (p_short > p_long)] = LABEL_SHORT
            scored = weighted_r_score(pred, pnl_long, pnl_short, sl, min_signals)
            e = float(scored["r_expectancy"])
            if e > best_e:
                best_e = e
                best = {"OPP": thr_opp, "LONG": thr_l, "SHORT": thr_s}
    return best


def _apply_combined(
    p_opp: np.ndarray,
    p_dir: np.ndarray,
    thr: Mapping[str, Optional[float]],
    eligible: np.ndarray,
) -> np.ndarray:
    pred = np.full(p_opp.shape[0], LABEL_NO_TRADE, dtype=np.int64)
    p_long, p_short = p_dir[:, LABEL_LONG], p_dir[:, LABEL_SHORT]
    opp_ok = eligible & (p_opp >= float(thr.get("OPP") or 0.0))
    if thr.get("LONG") is not None:
        pred[opp_ok & (p_long >= float(thr["LONG"])) & (p_long > p_short)] = LABEL_LONG
    if thr.get("SHORT") is not None:
        pred[opp_ok & (p_short >= float(thr["SHORT"])) & (p_short > p_long)] = LABEL_SHORT
    return pred


def _fold_arrays(
    labeled: pd.DataFrame,
    cols: Sequence[str],
    all_rows: np.ndarray,
    cost: float,
) -> Dict[str, np.ndarray]:
    feats = labeled.loc[:, list(cols)].to_numpy(dtype=np.float64)[all_rows]
    y = labeled["trade_label"].to_numpy(dtype=np.int64)[all_rows]
    opp = labeled["opp_label"].to_numpy(dtype=np.float64)[all_rows]
    dir_ok = labeled["dir_train_mask"].to_numpy(dtype=bool)[all_rows]
    sl = labeled["sl_pct"].to_numpy(dtype=np.float64)[all_rows]
    long_pnl = labeled["long_pnl_pct"].to_numpy(dtype=np.float64)[all_rows] - cost
    short_pnl = labeled["short_pnl_pct"].to_numpy(dtype=np.float64)[all_rows] - cost
    return {
        "feats": feats,
        "y": y,
        "opp": opp,
        "dir_ok": dir_ok,
        "sl": sl,
        "long_pnl": long_pnl,
        "short_pnl": short_pnl,
        "index": labeled.index.to_numpy()[all_rows] if hasattr(labeled.index, "to_numpy") else all_rows,
    }


def walk_forward_pair(
    labeled: pd.DataFrame,
    feature_cols: Sequence[str],
    wf_folds: Sequence[Tuple[np.ndarray, np.ndarray]],
    spec: Mapping[str, Any],
    config: Mapping[str, Any],
    inner_calibration_split: Callable[..., Tuple[np.ndarray, np.ndarray]],
    *,
    n_estimators: int,
    seed: int,
    skip_opp: bool,
) -> Dict[str, Any]:
    """Score one independent pair on expanding walk-forward folds only."""
    cols = [c for c in feature_cols if c in labeled.columns]
    if not cols:
        raise ValueError("No feature columns present for this family")
    all_rows = _usable_rows(labeled, cols)
    n = len(labeled)
    pack = _fold_arrays(labeled, cols, all_rows, float(config["round_trip_cost_pct"]))
    min_n = int(config.get("min_signals", 100))
    quantiles = list(config.get("signal_quantiles", [0.9, 0.8]))
    fold_r: List[float] = []
    fold_t: List[float] = []
    prec_l: List[float] = []
    prec_s: List[float] = []
    opp_aucs: List[float] = []
    dir_aucs: List[float] = []
    n_long_last = 0.0
    n_short_last = 0.0
    last_models: Dict[str, Any] = {}
    for fold_i, (tr_idx, va_idx) in enumerate(wf_folds, start=1):
        fit_idx, calib_idx = inner_calibration_split(
            tr_idx,
            calib_fraction=float(config.get("calib_fraction", 0.15)),
            purge_bars=int(config.get("purge_bars", FROZEN_HORIZON_BARS + 1)),
        )
        sel_fit = _selector(all_rows, fit_idx, n)
        sel_calib = _selector(all_rows, calib_idx, n)
        sel_eval = _selector(all_rows, va_idx, n)
        if not sel_fit.any() or not sel_eval.any() or not sel_calib.any():
            continue
        mean, std = _fit_scaler(pack["feats"][sel_fit])
        x_fit = _scale_x(pack["feats"][sel_fit], mean, std)
        opp_clf, dir_clf = _fit_opp_dir(
            x_fit,
            pack["opp"][sel_fit],
            pack["y"][sel_fit],
            pack["dir_ok"][sel_fit],
            config=config,
            n_estimators=n_estimators,
            seed=int(seed) + fold_i,
            skip_opp=skip_opp,
        )
        p_opp_c, p_dir_c = _predict_combined(
            opp_clf, dir_clf, _scale_x(pack["feats"][sel_calib], mean, std), skip_opp
        )
        p_opp_e, p_dir_e = _predict_combined(
            opp_clf, dir_clf, _scale_x(pack["feats"][sel_eval], mean, std), skip_opp
        )
        elig_c = _eligible_mask(labeled, all_rows[np.flatnonzero(sel_calib)], config)
        elig_e = _eligible_mask(labeled, all_rows[np.flatnonzero(sel_eval)], config)
        thr = _choose_combined_thresholds(
            p_opp_c,
            p_dir_c,
            pack["long_pnl"][sel_calib],
            pack["short_pnl"][sel_calib],
            pack["sl"][sel_calib],
            elig_c,
            quantiles=quantiles,
            min_signals=min_n,
            skip_opp=skip_opp,
        )
        pred = _apply_combined(p_opp_e, p_dir_e, thr, elig_e)
        scored = weighted_r_score(
            pred,
            pack["long_pnl"][sel_eval],
            pack["short_pnl"][sel_eval],
            pack["sl"][sel_eval],
            min_n,
        )
        fold_r.append(float(scored["r_expectancy"]))
        fold_t.append(float(scored["tstat"]))
        y_eval = pack["y"][sel_eval]
        prec_l.append(_class_precision(y_eval, pred, LABEL_LONG))
        prec_s.append(_class_precision(y_eval, pred, LABEL_SHORT))
        opp_aucs.append(_roc_auc((pack["opp"][sel_eval] >= 0.5).astype(int), p_opp_e))
        dec = pack["dir_ok"][sel_eval]
        if dec.any():
            dir_aucs.append(
                _roc_auc((y_eval[dec] == LABEL_LONG).astype(int), p_dir_e[dec, LABEL_LONG])
            )
        n_long_last = float(scored["n_LONG"])
        n_short_last = float(scored["n_SHORT"])
        last_models = {
            "opp_clf": opp_clf,
            "dir_clf": dir_clf,
            "mean": mean,
            "std": std,
            "thr": thr,
            "cols": cols,
        }
    finite_r = [s for s in fold_r if np.isfinite(s) and s > float("-inf") / 2]
    return {
        "wf_r_expectancy": float(np.mean(finite_r)) if finite_r else float("-inf"),
        "wf_tstat": float(np.nanmean(fold_t)) if fold_t else float("nan"),
        "wf_positive_share": (
            float(np.mean([s > 0.0 for s in finite_r])) if finite_r else float("nan")
        ),
        "wf_scores": fold_r,
        "n_LONG": n_long_last,
        "n_SHORT": n_short_last,
        "precision_LONG": float(np.nanmean(prec_l)) if prec_l else float("nan"),
        "precision_SHORT": float(np.nanmean(prec_s)) if prec_s else float("nan"),
        "opp_auc": float(np.nanmean(opp_aucs)) if opp_aucs else float("nan"),
        "dir_auc": float(np.nanmean(dir_aucs)) if dir_aucs else float("nan"),
        "n_features": len(cols),
        "models": last_models,
    }


def confirm_pair_test(
    labeled: pd.DataFrame,
    feature_cols: Sequence[str],
    splits: Mapping[str, np.ndarray],
    spec: Mapping[str, Any],
    config: Mapping[str, Any],
    *,
    n_estimators: int,
    seeds: Sequence[int],
    skip_opp: bool,
) -> Dict[str, Any]:
    """Fit on train, thresholds on val, score test once. Average finalist seeds."""
    cols = [c for c in feature_cols if c in labeled.columns]
    all_rows = _usable_rows(labeled, cols)
    n = len(labeled)
    pack = _fold_arrays(labeled, cols, all_rows, float(config["round_trip_cost_pct"]))
    sel_train = _selector(all_rows, splits["train"], n)
    sel_val = _selector(all_rows, splits["val"], n)
    sel_test = _selector(all_rows, splits["test"], n)
    min_n = int(config.get("min_signals", 100))
    quantiles = list(config.get("signal_quantiles", [0.9, 0.8]))
    seed_rows: List[Dict[str, Any]] = []
    for seed in seeds:
        mean, std = _fit_scaler(pack["feats"][sel_train])
        opp_clf, dir_clf = _fit_opp_dir(
            _scale_x(pack["feats"][sel_train], mean, std),
            pack["opp"][sel_train],
            pack["y"][sel_train],
            pack["dir_ok"][sel_train],
            config=config,
            n_estimators=n_estimators,
            seed=int(seed),
            skip_opp=skip_opp,
        )
        p_opp_v, p_dir_v = _predict_combined(
            opp_clf, dir_clf, _scale_x(pack["feats"][sel_val], mean, std), skip_opp
        )
        p_opp_t, p_dir_t = _predict_combined(
            opp_clf, dir_clf, _scale_x(pack["feats"][sel_test], mean, std), skip_opp
        )
        thr = _choose_combined_thresholds(
            p_opp_v,
            p_dir_v,
            pack["long_pnl"][sel_val],
            pack["short_pnl"][sel_val],
            pack["sl"][sel_val],
            _eligible_mask(labeled, all_rows[np.flatnonzero(sel_val)], config),
            quantiles=quantiles,
            min_signals=min_n,
            skip_opp=skip_opp,
        )
        pred = _apply_combined(
            p_opp_t,
            p_dir_t,
            thr,
            _eligible_mask(labeled, all_rows[np.flatnonzero(sel_test)], config),
        )
        scored = weighted_r_score(
            pred,
            pack["long_pnl"][sel_test],
            pack["short_pnl"][sel_test],
            pack["sl"][sel_test],
            min_n,
        )
        seed_rows.append(
            {
                "seed": int(seed),
                "test_r_expectancy": scored["r_expectancy"],
                "test_tstat": scored["tstat"],
                "test_precision_LONG": _class_precision(
                    pack["y"][sel_test], pred, LABEL_LONG
                ),
                "test_precision_SHORT": _class_precision(
                    pack["y"][sel_test], pred, LABEL_SHORT
                ),
                "test_n_LONG": scored["n_LONG"],
                "test_n_SHORT": scored["n_SHORT"],
                "n_test": int(sel_test.sum()),
                "thresholds": thr,
            }
        )
    r_vals = [float(s["test_r_expectancy"]) for s in seed_rows]
    finite = [v for v in r_vals if np.isfinite(v) and v > float("-inf") / 2]
    last = seed_rows[-1] if seed_rows else {}
    return {
        "seeds": seed_rows,
        "test_r_expectancy": float(np.mean(finite)) if finite else float("-inf"),
        "test_n_LONG": last.get("test_n_LONG", 0.0),
        "test_n_SHORT": last.get("test_n_SHORT", 0.0),
        "n_test": last.get("n_test", 0),
        "test_precision_LONG": last.get("test_precision_LONG", float("nan")),
        "test_precision_SHORT": last.get("test_precision_SHORT", float("nan")),
    }


def permutation_importance_r(
    labeled: pd.DataFrame,
    cols: Sequence[str],
    models: Mapping[str, Any],
    spec: Mapping[str, Any],
    config: Mapping[str, Any],
    eval_idx: np.ndarray,
    *,
    skip_opp: bool,
    rng: np.random.Generator,
) -> List[Dict[str, Any]]:
    """Permute one column at a time on WF eval features; no test rows."""
    if not models or (
        models.get("dir_clf") is None and models.get("opp_clf") is None
    ):
        return []
    all_rows = _usable_rows(labeled, cols)
    n = len(labeled)
    pack = _fold_arrays(labeled, cols, all_rows, float(config["round_trip_cost_pct"]))
    sel = _selector(all_rows, eval_idx, n)
    if not sel.any():
        return []
    mean, std = models["mean"], models["std"]
    x = _scale_x(pack["feats"][sel], mean, std)
    p_opp, p_dir = _predict_combined(
        models.get("opp_clf"), models.get("dir_clf"), x, skip_opp
    )
    elig = _eligible_mask(labeled, all_rows[np.flatnonzero(sel)], config)
    pred0 = _apply_combined(p_opp, p_dir, models["thr"], elig)
    base = weighted_r_score(
        pred0,
        pack["long_pnl"][sel],
        pack["short_pnl"][sel],
        pack["sl"][sel],
        int(config.get("min_signals", 100)),
    )["r_expectancy"]
    out: List[Dict[str, Any]] = []
    col_index = {c: i for i, c in enumerate(cols)}
    for col in cols:
        x_p = x.copy()
        x_p[:, col_index[col]] = rng.permutation(x_p[:, col_index[col]])
        p_o, p_d = _predict_combined(
            models.get("opp_clf"), models.get("dir_clf"), x_p, skip_opp
        )
        pred = _apply_combined(p_o, p_d, models["thr"], elig)
        scored = weighted_r_score(
            pred,
            pack["long_pnl"][sel],
            pack["short_pnl"][sel],
            pack["sl"][sel],
            int(config.get("min_signals", 100)),
        )
        out.append(
            {
                "column": col,
                "delta_r": float(base - scored["r_expectancy"])
                if np.isfinite(base) and np.isfinite(scored["r_expectancy"])
                else float("nan"),
            }
        )
    out.sort(key=lambda r: float(r["delta_r"]) if np.isfinite(r["delta_r"]) else -1e9, reverse=True)
    return out


def _print_stage_frame(name: str, rows: Sequence[Mapping[str, Any]]) -> None:
    """Print every row of a stage table (not just the top 3)."""
    print(f"\n===== {name} =====")
    if not rows:
        print("(empty)")
        return
    frame = pd.DataFrame([dict(r) for r in rows])
    nested = [
        c
        for c in frame.columns
        if frame[c].map(lambda v: isinstance(v, (dict, list))).any()
    ]
    show = frame.drop(columns=nested) if nested else frame
    with pd.option_context(
        "display.max_rows",
        None,
        "display.max_columns",
        None,
        "display.width",
        220,
    ):
        print(show.to_string(index=False))


def run_feature_label_sweep(
    featured: pd.DataFrame,
    *,
    config: Mapping[str, Any],
    compute_opportunity_labels: Callable[..., pd.DataFrame],
    summarize_label_gate: Callable[..., Dict[str, Any]],
    splits: Mapping[str, np.ndarray],
    wf_folds: Sequence[Tuple[np.ndarray, np.ndarray]],
    inner_calibration_split: Callable[..., Tuple[np.ndarray, np.ndarray]],
    export_dir: Path,
    n_estimators_stage_b: int = STAGE_B_TREES,
    n_estimators_stage_c: int = STAGE_C_TREES,
    top_k: int = STAGE_C_TOP_K,
) -> Dict[str, Any]:
    """Walk-forward screen, column search on finalists, one test look."""
    try:
        sys.stdout.reconfigure(line_buffering=True)
    except Exception:
        pass
    seed = int(config.get("seed", SCREEN_SEED))
    rng = np.random.default_rng(seed)
    trees = int(n_estimators_stage_b)
    stage_a: List[Dict[str, Any]] = []
    kept: List[Dict[str, Any]] = []
    labeled_by_id: Dict[str, pd.DataFrame] = {}
    skip_opp_by_id: Dict[str, bool] = {}
    for spec in LABEL_SPECS:
        frame = add_f7_features(featured, **spec_f7_kwargs(spec))
        labeled = apply_label_spec(
            frame,
            spec,
            compute_opportunity_labels,
            atr_period=int(config.get("atr_period", 14)),
            round_trip_cost_pct=float(config["round_trip_cost_pct"]),
        )
        valid = labeled["trade_label"].to_numpy() >= 0
        opp = labeled["opp_label"].to_numpy(dtype=np.float64)
        rate = float(np.nanmean(opp[valid])) if valid.any() else float("nan")
        degen = bool(np.asarray(labeled["opportunity_degenerate"]).reshape(-1)[0])
        heads = {"opportunity_rate": rate, "opportunity_degenerate": degen}
        report = summarize_label_gate(
            labeled,
            train_fraction=float(config["train_fraction"]),
            round_trip_cost_pct=float(config["round_trip_cost_pct"]),
            minority_rate=float(config["minority_rate"]),
            ambiguous_win_max=float(config["ambiguous_win_max"]),
            min_tp_cost_ratio=float(config["min_tp_cost_ratio"]),
        )
        report = {**report, **heads}
        keep = stage_a_should_keep(
            report,
            minority_rate=float(config["minority_rate"]),
            allow_degenerate_opp=str(spec.get("kind")) == "sign",
        )
        row = {
            "label_id": spec["id"],
            "label_group": spec.get("group", spec["id"]),
            "kind": spec["kind"],
            "horizon_bars": int(spec["horizon_bars"]),
            "kept": keep,
            **report,
        }
        stage_a.append(row)
        be = report.get("breakeven_win_rate")
        long_tp = report.get("base_long_tp_rate")
        short_tp = report.get("base_short_tp_rate")
        print(
            f"Stage A {spec['id']}: keep={keep} "
            f"opp={heads['opportunity_rate']:.3f} "
            f"degen={heads['opportunity_degenerate']} "
            f"BE={be} base_tp L/S={long_tp}/{short_tp}"
        )
        if keep:
            kept.append(dict(spec))
            labeled_by_id[spec["id"]] = labeled
            skip_opp_by_id[spec["id"]] = bool(heads["opportunity_degenerate"])

    col_sets = screen_column_sets()
    stage_b: List[Dict[str, Any]] = []
    models_by_pair: Dict[Tuple[str, str], Dict[str, Any]] = {}
    for spec in kept:
        skip_opp = skip_opp_by_id[spec["id"]]
        for fam, cols in col_sets:
            print(f"WF {spec['id']} x {fam} ({len(cols)} cols)")
            scored = walk_forward_pair(
                labeled_by_id[spec["id"]],
                cols,
                wf_folds,
                spec,
                config,
                inner_calibration_split,
                n_estimators=trees,
                seed=seed,
                skip_opp=skip_opp,
            )
            gate = next(r for r in stage_a if r["label_id"] == spec["id"])
            row = {
                "label_id": spec["id"],
                "label_group": spec.get("group", spec["id"]),
                "feature_family": fam,
                "kind": spec["kind"],
                "breakeven_win_rate": gate.get("breakeven_win_rate"),
                "opportunity_rate": gate.get("opportunity_rate"),
                "wf_r_expectancy": scored["wf_r_expectancy"],
                "wf_tstat": scored["wf_tstat"],
                "wf_positive_share": scored["wf_positive_share"],
                "n_LONG": scored["n_LONG"],
                "n_SHORT": scored["n_SHORT"],
                "precision_LONG": scored["precision_LONG"],
                "precision_SHORT": scored["precision_SHORT"],
                "opp_auc": scored["opp_auc"],
                "dir_auc": scored["dir_auc"],
                "n_features": scored["n_features"],
            }
            validate_stage_b_row(row)
            stage_b.append(row)
            models_by_pair[(spec["id"], fam)] = scored.get("models") or {}

    shuffle_best = float("-inf")
    shuffle_rows: List[Dict[str, Any]] = []
    for spec in kept:
        lab = labeled_by_id[spec["id"]].copy()
        lab["trade_label"] = block_shuffle(
            lab["trade_label"].to_numpy(), block=FROZEN_HORIZON_BARS, rng=rng
        )
        lab["opp_label"] = block_shuffle(
            lab["opp_label"].to_numpy(), block=FROZEN_HORIZON_BARS, rng=rng
        )
        lab["dir_train_mask"] = (lab["trade_label"].to_numpy() == LABEL_LONG) | (
            lab["trade_label"].to_numpy() == LABEL_SHORT
        )
        for fam, cols in col_sets:
            print(f"Shuffle WF {spec['id']} x {fam}")
            scored = walk_forward_pair(
                lab,
                cols,
                wf_folds,
                spec,
                config,
                inner_calibration_split,
                n_estimators=trees,
                seed=seed + 99,
                skip_opp=skip_opp_by_id[spec["id"]],
            )
            shuffle_rows.append(
                {
                    "label_id": spec["id"],
                    "feature_family": fam,
                    "wf_r_expectancy": scored["wf_r_expectancy"],
                }
            )
            val = float(scored["wf_r_expectancy"])
            if np.isfinite(val) and val > shuffle_best:
                shuffle_best = val

    top_pairs = select_top_pairs(stage_b, k=int(top_k))
    unique_specs: List[str] = []
    for pair in top_pairs:
        lid = str(pair["label_id"])
        if lid not in unique_specs:
            unique_specs.append(lid)
        if len(unique_specs) >= 2:
            break
    if not unique_specs and kept:
        unique_specs = [kept[0]["id"]]

    drop_rows: List[Dict[str, Any]] = []
    perm_rows: List[Dict[str, Any]] = []
    last_va = wf_folds[-1][1] if wf_folds else splits["val"]
    for spec_id in unique_specs:
        spec = label_spec_by_id(spec_id)
        spec_rows = [r for r in stage_b if r["label_id"] == spec_id]
        if not spec_rows:
            continue
        winner = select_top_pairs(spec_rows, k=1)[0]
        win_set = str(winner["feature_family"])
        win_cols = list(family_columns(win_set))
        skip_opp = skip_opp_by_id[spec_id]
        bases: List[Tuple[str, List[str]]] = [(win_set, win_cols)]
        if win_set != "F0" and win_set in FEATURE_FAMILY_ORDER:
            bases.append(("F0", list(family_columns("F0"))))
        for base_name, base_cols in bases:
            base_row = next(
                (r for r in spec_rows if r["feature_family"] == base_name),
                winner if base_name == win_set else None,
            )
            base_r = (
                float(base_row["wf_r_expectancy"])
                if base_row is not None
                else float("nan")
            )
            for drop in base_cols:
                left = [c for c in base_cols if c != drop]
                if len(left) < 2:
                    continue
                print(f"Drop-one {spec_id} {base_name} -{drop}")
                scored = walk_forward_pair(
                    labeled_by_id[spec_id],
                    left,
                    wf_folds,
                    spec,
                    config,
                    inner_calibration_split,
                    n_estimators=trees,
                    seed=seed,
                    skip_opp=skip_opp,
                )
                delta = (
                    float(base_r - scored["wf_r_expectancy"])
                    if np.isfinite(base_r) and np.isfinite(scored["wf_r_expectancy"])
                    else float("nan")
                )
                drop_rows.append(
                    {
                        "label_id": spec_id,
                        "dropped": drop,
                        "base_family": base_name,
                        "wf_r_expectancy": scored["wf_r_expectancy"],
                        "dir_auc": scored["dir_auc"],
                        "delta_r": delta,
                    }
                )
        models = models_by_pair.get((spec_id, win_set)) or {}
        perm_rows.extend(
            [
                {**r, "label_id": spec_id, "feature_family": win_set}
                for r in permutation_importance_r(
                    labeled_by_id[spec_id],
                    [c for c in win_cols if c in labeled_by_id[spec_id].columns],
                    models,
                    spec,
                    config,
                    last_va,
                    skip_opp=skip_opp,
                    rng=rng,
                )
            ]
        )

    stage_c: List[Dict[str, Any]] = []
    for pair in top_pairs:
        spec = label_spec_by_id(str(pair["label_id"]))
        cols = family_columns(str(pair["feature_family"]))
        print(f"Stage C {spec['id']} x {pair['feature_family']} seeds={FINALIST_SEEDS}")
        confirmed = confirm_pair_test(
            labeled_by_id[spec["id"]],
            cols,
            splits,
            spec,
            config,
            n_estimators=int(n_estimators_stage_c),
            seeds=FINALIST_SEEDS,
            skip_opp=skip_opp_by_id[spec["id"]],
        )
        try:
            wf_share = float(pair.get("wf_positive_share"))
        except (TypeError, ValueError):
            wf_share = 0.0
        try:
            wf_r = float(pair.get("wf_r_expectancy"))
        except (TypeError, ValueError):
            wf_r = float("nan")
        try:
            test_r = float(confirmed.get("test_r_expectancy"))
        except (TypeError, ValueError):
            test_r = float("nan")
        counts = bool(
            wf_share >= (2.0 / 3.0)
            and np.isfinite(wf_r)
            and wf_r > shuffle_best
            and np.isfinite(test_r)
            and test_r > 0.0
        )
        stage_c.append(
            {
                "label_id": spec["id"],
                "feature_family": pair["feature_family"],
                "wf_r_expectancy": pair.get("wf_r_expectancy"),
                "wf_positive_share": pair.get("wf_positive_share"),
                **{k: v for k, v in confirmed.items() if k != "seeds"},
                "seed_rows": confirmed["seeds"],
                "counts": counts,
            }
        )

    diagnosis = diagnose_sweep(stage_b, shuffle_best_r=shuffle_best)
    _print_stage_frame("Stage A", stage_a)
    _print_stage_frame("Stage B (walk-forward R)", stage_b)
    _print_stage_frame("Shuffle null", shuffle_rows)
    _print_stage_frame("Drop-one", drop_rows)
    _print_stage_frame("Permutation", perm_rows)
    _print_stage_frame("Stage C (test once)", stage_c)
    payload = {
        "resolution": str(config.get("resolution", FROZEN_RESOLUTION)),
        "horizon_bars": int(config.get("horizon_bars", FROZEN_HORIZON_BARS)),
        "model": "XGB",
        "n_estimators": trees,
        "stage_a": stage_a,
        "stage_a_kept": [s["id"] for s in kept],
        "stage_b": stage_b,
        "shuffle_best_r": shuffle_best,
        "shuffle_rows": shuffle_rows,
        "drop_one": drop_rows,
        "permutation": perm_rows,
        "top_pairs": top_pairs,
        "stage_c": stage_c,
        "diagnosis": diagnosis,
    }
    export_dir = Path(export_dir)
    export_dir.mkdir(parents=True, exist_ok=True)
    (export_dir / "sweep.json").write_text(
        json.dumps(payload, indent=2, default=float),
        encoding="utf-8",
    )
    print("wrote", export_dir / "sweep.json")
    print("diagnosis", diagnosis)
    return payload


In [ ]:
# CELL: FL_RUN
from pathlib import Path
import pandas as pd

_sweep_dir = Path(str(CONFIG["export_dir"]))
_sweep_dir.mkdir(parents=True, exist_ok=True)
sweep_payload = run_feature_label_sweep(
    featured,
    config=CONFIG,
    compute_opportunity_labels=compute_opportunity_labels,
    summarize_label_gate=summarize_label_gate,
    splits=splits,
    wf_folds=wf_folds,
    inner_calibration_split=inner_calibration_split,
    export_dir=_sweep_dir,
)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)


def _show_stage(name: str, rows: object) -> None:
    print(f"\n===== {name} =====")
    frame = pd.DataFrame(list(rows or []))
    if frame.empty:
        print("(empty)")
        return
    nested = [
        c
        for c in frame.columns
        if frame[c].map(lambda v: isinstance(v, (dict, list))).any()
    ]
    show = frame.drop(columns=nested) if nested else frame
    print(show.to_string(index=False))


_show_stage("Stage A", sweep_payload.get("stage_a"))
_show_stage("Stage B (walk-forward R)", sweep_payload.get("stage_b"))
_show_stage("Shuffle null", sweep_payload.get("shuffle_rows"))
_show_stage("Drop-one", sweep_payload.get("drop_one"))
_show_stage("Permutation", sweep_payload.get("permutation"))
_show_stage("Stage C (test once)", sweep_payload.get("stage_c"))
print("diagnosis", sweep_payload["diagnosis"])
print("kept labels", sweep_payload["stage_a_kept"])
print("shuffle_best_r", sweep_payload.get("shuffle_best_r"))
print("top pairs", sweep_payload.get("top_pairs"))
import shutil

_zip_path = shutil.make_archive(str(_sweep_dir), "zip", root_dir=str(_sweep_dir))
print("rewrote", _zip_path)
try:
    from google.colab import files

    files.download(str(_sweep_dir) + ".zip")
except ImportError:
    print("Not Colab; files remain under", _sweep_dir)


Stage A L0: keep=True opp=0.401 degen=False BE=0.3980841267886095 base_tp L/S=0.19790908489380005/0.19585787070733807
Stage A L1_2_2: keep=True opp=0.844 degen=False BE=0.5971261901829142 base_tp L/S=0.4227817111096407/0.4201349831271091
Stage A L1_3_3: keep=True opp=0.647 degen=False BE=0.5647507934552761 base_tp L/S=0.3221398795738768/0.31886455369549394


Stage A L1_1p5_1p5: keep=True opp=0.927 degen=False BE=0.6295015869105522 base_tp L/S=0.46377291073909876/0.46536094752861773
Stage A L2_2_1: keep=True opp=0.607 degen=False BE=0.46283492024388556 base_tp L/S=0.30311652219943097/0.3047376430887316


Stage A L2_3_1p5: keep=True opp=0.511 degen=False BE=0.41966772460703483 base_tp L/S=0.25431747502150465/0.25210084033613445
Stage A L3: keep=False opp=0.225 degen=False BE=0.3765005289701841 base_tp L/S=0.11192350956130484/0.11165883676305167


Stage A L4: keep=True opp=0.597 degen=False BE=0.4666666666666667 base_tp L/S=0.2993485234300076/0.29812493799398126
Stage A L5: keep=True opp=1.000 degen=True BE=0.6942523803658284 base_tp L/S=0.5190978537650055/0.48073679685174775
Stage A L6: keep=True opp=0.401 degen=False BE=0.3980841267886095 base_tp L/S=0.19790908489380005/0.19585787070733807


Stage A L7: keep=True opp=0.401 degen=False BE=0.3980841267886095 base_tp L/S=0.19790908489380005/0.19585787070733807
Stage A L8: keep=True opp=0.401 degen=False BE=0.3980841267886095 base_tp L/S=0.19790908489380005/0.19585787070733807
WF L0 x F0 (50 cols)


...

Stage C L0 x F0_minus_F4 seeds=(42, 43, 44)



===== Stage A =====
  label_id label_group       kind  horizon_bars  kept  passed  n_train_labeled  minority_rate  exclusive_long_expectancy_pct  exclusive_short_expectancy_pct  always_in_long_expectancy_pct  always_in_short_expectancy_pct  round_trip_cost_pct  median_tp_pct  median_sl_pct  tp_cost_ratio  min_tp_cost_ratio  breakeven_win_rate  base_long_tp_rate  base_short_tp_rate  long_ambiguous_share  short_ambiguous_share  tp_atr_mult  sl_atr_mult  opportunity_rate  opportunity_degenerate
        L0          L0    barrier            24  True    True            30226           0.15                       0.014104                        0.014872                      -0.001311                       -0.001858               0.0016       0.016473       0.008237      10.295884                4.0            0.398084           0.197909            0.195858              0.001669               0.001518          4.0          2.0          0.401478                   False
    L1_2_2          L1   


===== Stage A =====
  label_id label_group       kind  horizon_bars  kept  passed  n_train_labeled  minority_rate  exclusive_long_expectancy_pct  exclusive_short_expectancy_pct  always_in_long_expectancy_pct  always_in_short_expectancy_pct  round_trip_cost_pct  median_tp_pct  median_sl_pct  tp_cost_ratio  min_tp_cost_ratio  breakeven_win_rate  base_long_tp_rate  base_short_tp_rate  long_ambiguous_share  short_ambiguous_share  tp_atr_mult  sl_atr_mult  opportunity_rate  opportunity_degenerate
        L0          L0    barrier            24  True    True            30226           0.15                       0.014104                        0.014872                      -0.001311                       -0.001858               0.0016       0.016473       0.008237      10.295884                4.0            0.398084           0.197909            0.195858              0.001669               0.001518          4.0          2.0          0.401478                   False
    L1_2_2          L1   

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>